# Bangladesh Bank Circular / Guideline RAG Pipeline (Colab version)

This notebook builds a small **RAG (Retrieval-Augmented Generation)** pipeline for Bangladesh Bank
circulars/guidelines PDFs - structure-aware chunking (chapter/section), metadata tagging
(department, issue date, active/superseded status, referenced circulars), embeddings, and a
local vector DB (Chroma), with metadata-filtered retrieval.

**How to use this notebook:**
1. Run the cells in order (`Runtime > Run all`, or one by one).
2. When prompted, upload your PDF (e.g. `creditrisks.pdf`).
3. Edit the `doc_meta` dict in the "Ingest your PDF" cell to match your document.
4. Ask questions in the "Query" cell.
5. When a newer circular replaces an older one, use the "Mark superseded" cell.

## 1. Install dependencies

In [ ]:
!pip install -q pymupdf chromadb sentence-transformers rank-bm25 groq langchain-groq


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 62.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/

## 2. Imports and configuration

In [ ]:
import re
import sys
from datetime import datetime

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
DB_PATH = "/content/bb_vector_db_v2_exact_retrieval"
COLLECTION_NAME = "bb_documents"
MODEL_NAME = "BAAI/bge-large-en-v1.5"
MAX_CHUNK_CHARS = 1800
CHUNK_OVERLAP_CHARS = 150
MIN_CHUNK_CHARS = 60

CHAPTER_RE = re.compile(
    r'^(Chapter\s+\d{2}:\s*.+|Annexure-\d+)\s*$',
    re.MULTILINE
)
SECTION_RE = re.compile(
    r'^(\d\.\d+)\s+([A-Z][^\n]{2,90})\s*$',
    re.MULTILINE
)

CIRCULAR_REF_RE = re.compile(
    r'([A-Z]{2,10}(?:-\d+)?)\s+Circular(?:\s+Letter)?\s+No\.?\s*[-]?\s*(\d+)\s*,?\s*'
    r'(?:of|dated)\s+([0-9]{1,2}\s+\w+\s+[0-9]{4})',
    re.IGNORECASE
)

# Supports department codes such as FEPD-1, FE, BRPD, DOS, SDAD, etc.
DOC_HEADER_RE = re.compile(
    r'([A-Z]{2,10}(?:-\d+)?)\s+Circular(?:\s+Letter)?\s+No\.?\s*[-]?\s*(\d+)',
    re.IGNORECASE
)

DATE_RE = re.compile(
    r'\b('
    r'(?:'
        r'January|February|March|April|May|June|'
        r'July|August|September|October|November|December'
    r')'
    r'\s+'
    r'\d{1,2}'
    r'(?:,\s*|\s+)'
    r'\d{4}'
    r'|'
    r'\d{1,2}'
    r'\s+'
    r'(?:'
        r'January|February|March|April|May|June|'
        r'July|August|September|October|November|December'
    r')'
    r'\s+'
    r'\d{4}'
    r'|'
    r'\d{1,2}[/-]\d{1,2}[/-]\d{4}'
    r'|'
    r'\d{4}[/-]\d{1,2}[/-]\d{1,2}'
    r')\b',
    re.IGNORECASE
)

SUPERSESSION_HINTS = re.compile(
    r'(supersed|revis(ed|ion)|hereby cancel|in continuation to|stands (withdrawn|cancelled)|'
    r'shall replace|is amended)',
    re.IGNORECASE
)

STRONG_SUPERSESSION_HINTS = re.compile(
    r'(hereby cancel|stands (withdrawn|cancelled)|shall replace)',
    re.IGNORECASE
)

PARA_RE = re.compile(
    r'^(\d{1,2})\.\s+(?=[A-Z\(])',
    re.MULTILINE
)

DOC_TYPE_PATTERNS = [
    ("master_circular", re.compile(r'master\s+circular', re.IGNORECASE)),
    ("circular_letter", re.compile(r'circular\s+letter', re.IGNORECASE)),
    ("circular", re.compile(r'\bcircular\b', re.IGNORECASE)),
    ("guideline", re.compile(r'\bguideline(s)?\b', re.IGNORECASE)),
    ("notification", re.compile(r'\bnotification\b', re.IGNORECASE)),
    ("press_release", re.compile(r'press\s+release', re.IGNORECASE)),
]

# Question-side identifier extraction.
QUERY_CIRCULAR_RE = re.compile(
    r'circular\s*(?:no\.?|number)\s*[-:]?\s*(\d+)\b',
    re.IGNORECASE
)

QUERY_DATE_RE = re.compile(
    r'\b('
    r'(?:January|February|March|April|May|June|July|August|September|October|November|December)'
    r'\s+\d{1,2},?\s+\d{4}'
    r'|'
    r'\d{1,2}\s+'
    r'(?:January|February|March|April|May|June|July|August|September|October|November|December)'
    r'\s+\d{4}'
    r')\b',
    re.IGNORECASE
)


## 3. Text extraction and structure detection

In [ ]:
import pymupdf
def extract_text(pdf_path: str) -> str:
    doc = pymupdf.open(pdf_path)

    pages = []

    for page_num, page in enumerate(doc, start=1):
        text = page.get_text().strip()

        if text:
            pages.append(f"[PAGE {page_num}]\n{text}")

    doc.close()

    return "\n".join(pages)

def detect_doc_type(text, pdf_name=""):
    t = text.lower()
    name = pdf_name.lower()

    if "monetary policy statement" in t or "monetary policy" in name:
        return "monetary_policy"

    if "circular no." in t or "circular" in name:
        return "circular"

    if "annual report" in t or "annual report" in name:
        return "report"

    return "general"


def find_structure(text: str):
    """Return sorted list of (position, level, label) markers, trying progressively
    looser structural conventions -- BB documents vary a lot in how they're structured:

      Tier 1: Chapter/Section headings ("Chapter 01:", "1.1 Title")
              -- used in master circulars and comprehensive guidelines.
      Tier 2: Numbered paragraphs ("01.", "02.", "03." at the start of a line)
              -- used in most ordinary circulars and circular letters, which don't
              have chapter headings at all.

    If neither tier finds anything, split_into_sections() falls back to treating the
    whole document as one section, so no document type ever produces zero chunks.
    """
    markers = []
    for m in CHAPTER_RE.finditer(text):
        markers.append((m.start(), "chapter", m.group(1).strip()))
    for m in SECTION_RE.finditer(text):
        markers.append((m.start(), "section", f"{m.group(1)} {m.group(2)}".strip()))

    if not markers:
        for m in PARA_RE.finditer(text):
            markers.append((m.start(), "section", f"Para {m.group(1)}"))

    markers.sort(key=lambda x: x[0])
    return markers


def detect_referenced_circulars(text: str):
    """Pull out any other-circular references mentioned inside this document."""
    refs = set()
    for dept, num, date_raw in CIRCULAR_REF_RE.findall(text):
        date_clean = re.sub(r'\s+', ' ', date_raw).strip()
        refs.add(f"{dept.upper()} Circular No. {num} of {date_clean}")
    return sorted(refs)


def detect_supersession_signal(text: str) -> bool:
    """Flag for human review: does this doc's language suggest it replaces an earlier one?"""
    return bool(SUPERSESSION_HINTS.search(text[:3000]))  # preamble usually says it early


def classify_document_type(text: str) -> str:
    """Guess the BB document type from its own header text, so document_type doesn't
    have to be typed in by hand and can later be used to filter queries
    (e.g. 'only search master circulars')."""
    head = text[:800]
    for label, pattern in DOC_TYPE_PATTERNS:
        if pattern.search(head):
            return label
    return "other"


## 4. Structure-aware chunking (chapter/section boundaries, not blind word-count blocks)

In [ ]:
def split_into_sections(text: str, markers):
    sections = []
    current_chapter = "Preamble"
    current_section = None

    for i, (pos, level, label) in enumerate(markers):

        end = markers[i + 1][0] if i + 1 < len(markers) else len(text)

        body = text[pos:end].strip()

        if not body:
            continue

        if level == "chapter":
            current_chapter = label
            current_section = None

        elif level == "section":
            current_section = label

        sections.append({
            "chapter": current_chapter,
            "section": current_section,
            "text": body
        })

    # Text before first detected heading
    if markers:
        preamble = text[:markers[0][0]].strip()

        if len(preamble) > 200:
            sections.insert(
                0,
                {
                    "chapter": "Preamble",
                    "section": None,
                    "text": preamble
                }
            )

    # IMPORTANT:
    # If no structure was detected, don't return zero sections.
    if not sections and text.strip():
        sections.append({
            "chapter": "Full Document",
            "section": None,
            "text": text.strip()
        })

    return sections


def chunk_text(text, doc_type,
                        max_chars=1800,
                        overlap=200):

    if doc_type == "monetary_policy":
        max_chars = 2200
        overlap = 250

    elif doc_type == "circular":
        max_chars = 1600
        overlap = 150

    elif doc_type == "report":
        max_chars = 2000
        overlap = 200

    else:
        max_chars = 1800
        overlap = 200

    paragraphs = [
        p.strip()
        for p in text.split("\n")
        if p.strip()
    ]

    chunks = []
    current = ""

    for para in paragraphs:

        # Paragraph fits
        if len(current) + len(para) + 1 <= max_chars:

            current += para + "\n"

        else:

            if current.strip():
                chunks.append(current.strip())

            # overlap from previous chunk
            tail = current[-overlap:] if current else ""

            current = tail + para + "\n"

    if current.strip():
        chunks.append(current.strip())

    return chunks


def build_chunks(
    sections,
    doc_meta,
    referenced_circulars,
    supersession_flag,

    doc_type="guideline"
):

    records = []

    for sec_idx, sec in enumerate(sections):

        text_chunks = chunk_text(
            sec["text"],
            doc_type
        )

        for ch_idx, chunk in enumerate(text_chunks):

            if len(chunk.strip()) < MIN_CHUNK_CHARS:
                continue

            header = sec["chapter"]

            if sec["section"]:
                header += f" > {sec['section']}"

            prefixed_text = (
                f"[{header}]\n"
                f"{chunk}"
            )

            meta = dict(doc_meta)

            # Recover PDF page range from the extracted [PAGE N] markers.
            # This makes the source page available during retrieval and evaluation.
            page_numbers = [
                int(p)
                for p in re.findall(r"\[PAGE\s+(\d+)\]", chunk)
            ]

            meta.update({
                "chapter": sec["chapter"],
                "section": sec["section"] or "",

                "document_type": doc_type,

                "referenced_circulars":
                    ", ".join(referenced_circulars)
                    if referenced_circulars else "",

                "needs_supersession_review":
                    bool(supersession_flag),

                "page_start":
                    min(page_numbers) if page_numbers else "",

                "page_end":
                    max(page_numbers) if page_numbers else "",
            })

            records.append({
                "id":
                    f"{doc_meta['doc_id']}_"
                    f"s{sec_idx}_c{ch_idx}",

                "text": prefixed_text,

                "metadata": meta
            })

    return records

## 5. Embedding model and local vector DB (Chroma)

In [ ]:
_embedder = None
_collection = None

def get_embedder():
    global _embedder
    if _embedder is None:
        from sentence_transformers import SentenceTransformer
        print("Loading embedding model (first call downloads it)...")
        _embedder = SentenceTransformer(MODEL_NAME)
    return _embedder


def get_collection():
    global _collection

    if _collection is None:
        import chromadb

        client = chromadb.PersistentClient(path=DB_PATH)

        _collection = client.get_or_create_collection(
            name=COLLECTION_NAME,
            metadata={"hnsw:space": "cosine"}
        )

    return _collection

## 6. Ingest a PDF

In [ ]:
import os
import re

MIN_EXTRACTED_CHARS = 200

def extract_document_metadata(text: str, pdf_path: str):
    header = text[:5000]
    lower_text = header.lower()
    pdf_name = os.path.basename(pdf_path)
    pdf_name_lower = pdf_name.lower()

    # Payment reports must be detected before generic "circular".
    if (
        "payment systems report" in lower_text
        or "payment system report" in lower_text
        or "paymentreport" in pdf_name_lower
    ):
        document_type = "Payment Systems Report"
    elif "monetary policy statement" in lower_text:
        document_type = "Monetary Policy Statement"
    elif (
        "guidelines on credit risk management" in lower_text
        or "guideline on credit risk management" in lower_text
    ):
        document_type = "Guideline"
    elif "guideline" in lower_text:
        document_type = "Guideline"
    elif "master circular" in lower_text:
        document_type = "Master Circular"
    elif "circular letter" in lower_text:
        document_type = "Circular Letter"
    elif "circular" in lower_text:
        document_type = "Circular"
    elif "report" in lower_text:
        document_type = "Report"
    else:
        document_type = "Unknown"

    circular_number = ""
    for pattern in [
        r'Circular\s+No\.?\s*[:\-]?\s*([A-Za-z0-9\-/]+)',
        r'Circular\s+No\.?\s*([0-9]+)',
    ]:
        match = re.search(pattern, header, re.IGNORECASE)
        if match:
            circular_number = match.group(1).strip()
            break

    issue_date = ""
    for pattern in [
        r'Date\s*[:\-]\s*([A-Za-z]+\s+\d{1,2},\s+\d{4})',
        r'Date\s*[:\-]\s*(\d{1,2}\s+[A-Za-z]+\s+\d{4})',
        r'([A-Z][a-z]+\s+\d{1,2},\s+\d{4})',
        r'(\d{1,2}\s+[A-Z][a-z]+\s+\d{4})',
        r'(\d{1,2}[\/-]\d{1,2}[\/-]\d{4})',
    ]:
        match = re.search(pattern, header, re.IGNORECASE)
        if match:
            issue_date = match.group(1).strip()
            break

    year_match = re.search(r'\b(20\d{2})\b', pdf_name)
    if not year_match:
        year_match = re.search(r'\b(20\d{2})\b', header)
    year = year_match.group(1) if year_match else ""

    reporting_period = ""
    if "jun" in pdf_name_lower or "june" in pdf_name_lower:
        reporting_period = f"H1 {year}" if year else "H1"
    elif "dec" in pdf_name_lower or "december" in pdf_name_lower:
        reporting_period = f"Annual {year}" if year else "Annual"

    period_match = re.search(
        r'\b(H1|H2)\s*(?:FY)?\s*(20\d{2}|\d{2})\b',
        pdf_name + " " + header,
        re.IGNORECASE
    )
    if period_match:
        half = period_match.group(1).upper()
        pyear = period_match.group(2)
        if len(pyear) == 2:
            pyear = "20" + pyear
        reporting_period = f"{half} {pyear}"
        if not year:
            year = pyear

    base = os.path.splitext(pdf_name)[0]
    if document_type in ["Payment Systems Report", "Monetary Policy Statement", "Report"]:
        document_id = re.sub(r'[^A-Za-z0-9]+', '-', base).strip('-').upper()
    elif circular_number:
        document_id = f"CIRCULAR-{circular_number}"
    else:
        document_id = re.sub(r'[^A-Za-z0-9]+', '-', base).strip('-').upper()

    return {
        "pdf_name": pdf_name,
        "doc_id": document_id,
        "document_type": document_type,
        "circular_number": circular_number,
        "issue_date": issue_date,
        "year": year,
        "reporting_period": reporting_period,
    }

def ingest_pdf(pdf_path: str, doc_meta: dict):
    doc_meta = dict(doc_meta)
    doc_meta.setdefault("is_active", True)
    doc_meta.setdefault("ingested_at", datetime.utcnow().isoformat())
    doc_meta.setdefault("department", "")
    doc_meta.setdefault("issue_date", "")
    doc_meta.setdefault("document_type", "Unknown")
    doc_meta.setdefault("year", "")
    doc_meta.setdefault("reporting_period", "")

    print(f"[1/5] Extracting text from {pdf_path} ...")
    text = extract_text(pdf_path)
    print(f"      {len(text):,} characters extracted")

    document_meta = extract_document_metadata(text, pdf_path)
    doc_meta.update(document_meta)

    print("      PDF Name       :", doc_meta.get("pdf_name", "N/A"))
    print("      Document ID    :", doc_meta.get("doc_id", "N/A"))
    print("      Document Type  :", doc_meta.get("document_type", "N/A"))
    print("      Year           :", doc_meta.get("year", "N/A"))
    print("      Reporting      :", doc_meta.get("reporting_period", "N/A"))
    print("      Circular No.   :", doc_meta.get("circular_number", "N/A"))
    print("      Issue Date     :", doc_meta.get("issue_date", "N/A"))

    if len(text.strip()) < MIN_EXTRACTED_CHARS:
        print("      WARNING: very little text extracted; OCR may be required.")

    print("[2/5] Detecting chapter/section/paragraph structure ...")
    markers = find_structure(text)
    print(f"      {len(markers)} structural markers found")

    referenced = detect_referenced_circulars(text)
    supersession_flag = detect_supersession_signal(text)

    print("[3/5] Splitting into structure-aware chunks ...")
    sections = split_into_sections(text, markers)
    records = build_chunks(
        sections,
        doc_meta,
        referenced,
        supersession_flag,
        doc_meta.get("document_type", "Unknown")
    )
    print(f"      {len(records)} chunks created")

    print("[4/5] Loading embedding model ...")
    model = get_embedder()

    print("[5/5] Embedding + writing to local vector DB ...")
    collection = get_collection()
    batch = 64

    for i in range(0, len(records), batch):
        group = records[i:i + batch]
        embeddings = model.encode(
            [r["text"] for r in group],
            normalize_embeddings=True
        ).tolist()

        # upsert makes repeated ingestion safe
        collection.upsert(
            ids=[r["id"] for r in group],
            embeddings=embeddings,
            documents=[r["text"] for r in group],
            metadatas=[r["metadata"] for r in group],
        )

    print(
        f"Done. Ingested '{doc_meta['doc_id']}' "
        f"({doc_meta.get('pdf_name', 'unknown PDF')}) "
        f"-> {len(records)} chunks into '{DB_PATH}'."
    )


## 7. Query with metadata filtering (defaults to only currently-active docs)

In [ ]:
def extract_query_constraints(question: str):
    q = question.lower()

    circular_match = QUERY_CIRCULAR_RE.search(question)
    date_match = QUERY_DATE_RE.search(question)

    circular_number = circular_match.group(1).strip() if circular_match else ""
    requested_date = date_match.group(1).strip() if date_match else ""

    year_matches = re.findall(r"\b(20\d{2})\b", q)
    requested_year = year_matches[0] if year_matches else ""

    fiscal_year = ""
    fy_match = re.search(
        r'\b(?:h1|h2)?\s*fy\s*(\d{2,4})\b',
        q,
        re.IGNORECASE
    )
    if fy_match:
        fiscal_year = fy_match.group(1)
        if len(fiscal_year) == 2:
            fiscal_year = "20" + fiscal_year

    quarter = ""
    quarter_match = re.search(r'\bq([1-4])\b', q)
    if quarter_match:
        quarter = "Q" + quarter_match.group(1)

    half = ""
    half_match = re.search(r'\b(h1|h2)\b', q, re.IGNORECASE)
    if half_match:
        half = half_match.group(1).upper()
    if "first half" in q or "first-half" in q:
        half = "H1"
    elif "second half" in q or "second-half" in q:
        half = "H2"

    full_year = (
        "full year" in q
        or "full-year" in q
        or "annual" in q
    )

    return {
        "circular_number": circular_number,
        "requested_date": requested_date,
        "requested_year": requested_year,
        "fiscal_year": fiscal_year,
        "quarter": quarter,
        "half": half,
        "full_year": full_year,
    }


def _metadata_relevance_score(question, meta):
    constraints = extract_query_constraints(question)

    searchable = " ".join([
        str(meta.get("pdf_name", "")),
        str(meta.get("doc_id", "")),
        str(meta.get("issue_date", "")),
        str(meta.get("year", "")),
        str(meta.get("reporting_period", "")),
        str(meta.get("circular_number", "")),
    ]).lower()

    document_type = str(
        meta.get("document_type", "")
    ).lower()

    bonus = 0.0

    requested_year = constraints["requested_year"]
    if requested_year:
        if requested_year in searchable:
            bonus += 0.80
        else:
            bonus -= 1.00

    fiscal_year = constraints["fiscal_year"]
    if fiscal_year:
        if (
            fiscal_year in searchable
            or ("fy" + fiscal_year[-2:]) in searchable
            or ("fy" + fiscal_year) in searchable
        ):
            bonus += 0.60
        else:
            bonus -= 0.70

    quarter = constraints["quarter"]
    if quarter:
        if quarter.lower() in searchable:
            bonus += 0.35
        elif (
            quarter == "Q4"
            and (
                "dec" in searchable
                or "december" in searchable
                or "annual" in str(
                    meta.get("reporting_period", "")
                ).lower()
            )
        ):
            bonus += 0.30
        else:
            bonus -= 0.20

    half = constraints["half"]
    if half:
        if half.lower() in searchable:
            bonus += 0.35
        elif (
            half == "H1"
            and ("jun" in searchable or "june" in searchable)
        ):
            bonus += 0.30
        elif (
            half == "H2"
            and ("dec" in searchable or "december" in searchable)
        ):
            bonus += 0.30
        else:
            bonus -= 0.20

    if constraints["full_year"]:
        if (
            "annual" in str(meta.get("reporting_period", "")).lower()
            or "dec" in searchable
            or "december" in searchable
        ):
            bonus += 0.25

    requested_circular = constraints["circular_number"]
    if requested_circular:
        if requested_circular.lower() in searchable:
            bonus += 0.80
        else:
            bonus -= 1.00

    payment_terms = [
        "payment", "npsb", "ibft", "psp", "mfs",
        "transaction", "atm", "pos", "payment platform"
    ]
    if any(t in question.lower() for t in payment_terms):
        if "payment systems report" in document_type:
            bonus += 0.25

    return bonus


def query(question, n_results=10, candidate_pool=100):
    model = get_embedder()
    collection = get_collection()

    query_embedding = model.encode(
        question,
        normalize_embeddings=True
    ).tolist()

    total = collection.count()

    if total == 0:
        return {
            "documents": [[]],
            "metadatas": [[]],
            "distances": [[]],
            "similarities": [[]],
            "rerank_scores": [[]],
            "metadata_bonuses": [[]],
        }

    fetch_k = min(
        max(candidate_pool, n_results),
        total
    )

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=fetch_k,
        where={"is_active": True},
        include=["documents", "metadatas", "distances"]
    )

    documents = results.get("documents", [[]])[0]
    metadatas = results.get("metadatas", [[]])[0]
    distances = results.get("distances", [[]])[0]

    constraints = extract_query_constraints(question)
    requested_year = constraints["requested_year"]

    candidates = []

    for doc, meta, distance in zip(
        documents,
        metadatas,
        distances
    ):
        metadata_text = " ".join([
            str(meta.get("pdf_name", "")),
            str(meta.get("doc_id", "")),
            str(meta.get("issue_date", "")),
            str(meta.get("year", "")),
            str(meta.get("reporting_period", "")),
        ]).lower()

        # Hard year constraint when the user explicitly names a year.
        if requested_year and requested_year not in metadata_text:
            continue

        semantic_similarity = 1.0 - float(distance)
        metadata_bonus = _metadata_relevance_score(
            question,
            meta
        )

        rerank_score = (
            0.70 * semantic_similarity
            + 0.30 * metadata_bonus
        )

        candidates.append({
            "document": doc,
            "metadata": meta,
            "distance": float(distance),
            "semantic_similarity": semantic_similarity,
            "metadata_bonus": metadata_bonus,
            "rerank_score": rerank_score,
        })

    # Safety fallback only if no year-matching candidate exists.
    if not candidates:
        for doc, meta, distance in zip(
            documents, metadatas, distances
        ):
            semantic_similarity = 1.0 - float(distance)
            metadata_bonus = _metadata_relevance_score(
                question,
                meta
            )
            rerank_score = (
                0.70 * semantic_similarity
                + 0.30 * metadata_bonus
            )
            candidates.append({
                "document": doc,
                "metadata": meta,
                "distance": float(distance),
                "semantic_similarity": semantic_similarity,
                "metadata_bonus": metadata_bonus,
                "rerank_score": rerank_score,
            })

    candidates.sort(
        key=lambda x: x["rerank_score"],
        reverse=True
    )

    top = candidates[:n_results]

    return {
        "documents": [[x["document"] for x in top]],
        "metadatas": [[x["metadata"] for x in top]],
        "distances": [[x["distance"] for x in top]],
        "similarities": [[x["semantic_similarity"] for x in top]],
        "rerank_scores": [[x["rerank_score"] for x in top]],
        "metadata_bonuses": [[x["metadata_bonus"] for x in top]],
    }


In [ ]:
def assess_retrieval_confidence(similarities):
    """
    Confidence based on the actual cosine similarity ranking.
    This is retrieval confidence, NOT answer correctness.
    """
    if not similarities:
        return {
            "status": "NO_EVIDENCE",
            "score": 0.0,
            "margin": 0.0
        }

    top_similarity = float(similarities[0])

    if len(similarities) >= 2:
        margin = float(
            similarities[0] - similarities[1]
        )
    else:
        margin = 0.0

    if top_similarity >= 0.80 and margin >= 0.05:
        status = "HIGH"
    elif top_similarity >= 0.65:
        status = "MEDIUM"
    else:
        status = "LOW"

    return {
        "status": status,
        "score": top_similarity,
        "margin": margin
    }


## 8. Mark a document as superseded when a newer circular replaces it

In [ ]:
def mark_superseded(old_doc_id: str, new_doc_id: str):
    collection = get_collection()
    existing = collection.get(where={"doc_id": old_doc_id})
    if not existing["ids"]:
        print(f"No chunks found for doc_id={old_doc_id}")
        return
    updated_metas = []
    for meta in existing["metadatas"]:
        meta["is_active"] = False
        meta["superseded_by"] = new_doc_id
        updated_metas.append(meta)
    collection.update(ids=existing["ids"], metadatas=updated_metas)
    print(f"Marked {len(existing['ids'])} chunks of '{old_doc_id}' as superseded by '{new_doc_id}'.")

## 8b. Auto-derive metadata from the PDF itself, and auto-flag likely supersession links
Instead of typing `doc_id`/`department`/`issue_date` by hand for every file, pull them straight out of each PDF's own header text. This also scans every ingested doc's `referenced_circulars` for candidates that likely supersede an earlier one, so you don't have to track version history in your head.

In [ ]:
def auto_doc_meta(text: str, pdf_path: str) -> dict:
    # Use exactly the same metadata extractor as ingestion.
    return extract_document_metadata(text, pdf_path)


def auto_link_supersession():
    collection = get_collection()
    all_meta = collection.get()["metadatas"]

    by_doc = {}
    for meta in all_meta:
        by_doc.setdefault(meta.get("doc_id", ""), meta)

    any_candidates = False

    for doc_id, meta in by_doc.items():
        if not meta.get("needs_supersession_review"):
            continue

        refs = meta.get("referenced_circulars", "")
        if not refs:
            continue

        for other_id in by_doc:
            if other_id == doc_id:
                continue

            parts = other_id.split("-")
            if len(parts) < 2:
                continue

            dept, num = parts[0], parts[1]

            if dept in refs and num in refs:
                any_candidates = True
                print(
                    f"[candidate] '{doc_id}' may supersede/amend "
                    f"'{other_id}'"
                )

    if not any_candidates:
        print(
            "No supersession candidates found among "
            "currently ingested documents."
        )


## 9. Upload your PDFs
Run this cell, then select **all** your circular PDFs at once (multi-select in the file dialog). Metadata is auto-detected from each PDF's header in the next cell.

In [ ]:
import os
from google.colab import files

uploaded = files.upload()
pdf_paths = list(uploaded.keys())

print("\n" + "=" * 80)
print("PDF METADATA CHECK")
print("=" * 80)

for pdf_path in pdf_paths:
    preview_text = extract_text(pdf_path)
    doc_meta = extract_document_metadata(preview_text, pdf_path)
    print("\nPDF:", pdf_path)
    print("Document Type   :", doc_meta.get("document_type"))
    print("Year            :", doc_meta.get("year"))
    print("Reporting Period:", doc_meta.get("reporting_period"))
    print("Document ID     :", doc_meta.get("doc_id"))

print("\n" + "=" * 80)
print("INGESTING DOCUMENTS")
print("=" * 80)

for pdf_path in pdf_paths:
    preview_text = extract_text(pdf_path)
    doc_meta = extract_document_metadata(preview_text, pdf_path)
    ingest_pdf(pdf_path, doc_meta)

# BM25 is built lazily from the current Chroma collection.
# Reset it after ingestion so the new documents are included.
_BM25_INDEX = None
_BM25_RECORDS = []

print("\nAll uploaded PDFs have been ingested.")


Saving creditrisks.pdf to creditrisks.pdf
Saving jul162026fepd-121e.pdf to jul162026fepd-121e.pdf
Saving jul222026fepd-122e.pdf to jul222026fepd-122e.pdf
Saving jul222026fepd-123e.pdf to jul222026fepd-123e.pdf
Saving jul232026fepd-124e.pdf to jul232026fepd-124e.pdf
Saving jul292026fepd-125e.pdf to jul292026fepd-125e.pdf
Saving jul302026fepd-126e.pdf to jul302026fepd-126e.pdf
Saving mps_h1fy25.pdf to mps_h1fy25.pdf
Saving mps_h1fy26.pdf to mps_h1fy26.pdf
Saving mps_h1fy27_new.pdf to mps_h1fy27_new.pdf
Saving mps_h2fy24.pdf to mps_h2fy24.pdf
Saving mps_h2fy25.pdf to mps_h2fy25.pdf
Saving mps_h2fy26.pdf to mps_h2fy26.pdf
Saving paymentreport_dec2021.pdf to paymentreport_dec2021.pdf
Saving paymentreport_dec2024.pdf to paymentreport_dec2024.pdf
Saving paymentreport_dec2025.pdf to paymentreport_dec2025.pdf
Saving paymentreport_july2019.pdf to paymentreport_july2019.pdf
Saving paymentreport_jun2024.pdf to paymentreport_jun2024.pdf

PDF METADATA CHECK

PDF: creditrisks.pdf
Document Type   : Gu

/tmp/ipykernel_2830/2516422492.py:108: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  doc_meta.setdefault("ingested_at", datetime.utcnow().isoformat())


[1/5] Extracting text from creditrisks.pdf ...
      112,763 characters extracted
      PDF Name       : creditrisks.pdf
      Document ID    : CREDITRISKS
      Document Type  : Guideline
      Year           : 2016
      Reporting      : 
      Circular No.   : 
      Issue Date     : March 08, 2016
[2/5] Detecting chapter/section/paragraph structure ...
      56 structural markers found
[3/5] Splitting into structure-aware chunks ...
      100 chunks created
[4/5] Loading embedding model ...
Loading embedding model (first call downloads it)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

[5/5] Embedding + writing to local vector DB ...
Done. Ingested 'CREDITRISKS' (creditrisks.pdf) -> 100 chunks into '/content/bb_vector_db_v2_exact_retrieval'.
[1/5] Extracting text from jul162026fepd-121e.pdf ...
      4,809 characters extracted
      PDF Name       : jul162026fepd-121e.pdf
      Document ID    : CIRCULAR-21
      Document Type  : Circular
      Year           : 2026
      Reporting      : 
      Circular No.   : 21
      Issue Date     : July 16, 2026
[2/5] Detecting chapter/section/paragraph structure ...
      6 structural markers found
[3/5] Splitting into structure-aware chunks ...
      7 chunks created
[4/5] Loading embedding model ...
[5/5] Embedding + writing to local vector DB ...


/tmp/ipykernel_2830/2516422492.py:108: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  doc_meta.setdefault("ingested_at", datetime.utcnow().isoformat())


Done. Ingested 'CIRCULAR-21' (jul162026fepd-121e.pdf) -> 7 chunks into '/content/bb_vector_db_v2_exact_retrieval'.
[1/5] Extracting text from jul222026fepd-122e.pdf ...
      6,609 characters extracted
      PDF Name       : jul222026fepd-122e.pdf
      Document ID    : CIRCULAR-22
      Document Type  : Guideline
      Year           : 2026
      Reporting      : 
      Circular No.   : 22
      Issue Date     : July 22, 2026
[2/5] Detecting chapter/section/paragraph structure ...
      10 structural markers found
[3/5] Splitting into structure-aware chunks ...
      11 chunks created
[4/5] Loading embedding model ...
[5/5] Embedding + writing to local vector DB ...
Done. Ingested 'CIRCULAR-22' (jul222026fepd-122e.pdf) -> 11 chunks into '/content/bb_vector_db_v2_exact_retrieval'.
[1/5] Extracting text from jul222026fepd-123e.pdf ...
      1,004 characters extracted
      PDF Name       : jul222026fepd-123e.pdf
      Document ID    : CIRCULAR-23
      Document Type  : Circular
      Ye

### Important: re-ingest after changing metadata\nThe new `circular_number` metadata is stored only when PDFs are ingested. If your existing Chroma collection was created before this fix, clear/recreate the collection and upload the PDFs again before testing exact Circular No. questions.\n

## Important: rebuild the vector DB after this fix

The notebook now stores `page_start`/`page_end` metadata and uses metadata-aware reranking. If your existing Chroma collection was created before this change, **clear/recreate the collection and re-ingest all PDFs** before testing. Otherwise old chunks will not have the new metadata.

For example, in Colab, restart the runtime or delete `/content/bb_vector_db`, then run the notebook from the beginning and upload all PDFs again.


In [ ]:
!pip install groq

In [ ]:
!pip install -q langchain-groq

In [ ]:
import getpass, os
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API key: ")

llm = ChatGroq(
    model="qwen/qwen3.8-27B",
    temperature=0,
)


Groq API key: ··········


In [ ]:
def build_prompt(context, question):

    return f"""
You are a Bangladesh Bank document question-answering system.

Answer the question using ONLY the retrieved Bangladesh Bank evidence.

STRICT OUTPUT RULES:

- Output ONLY the final answer.
- Do NOT show your reasoning.
- Do NOT explain how you reached the answer.
- Do NOT show analysis, planning, checking, verification, drafting, or self-correction.
- Do NOT output <think> or </think>.
- Do NOT output "thinking process".
- Do NOT output "final answer generation".
- Do NOT output "proceeds", "done", "matches evidence", or similar internal text.
- Do NOT output statements such as "I will", "I need to", "let me check", or "I should".
- Do NOT mention the retrieved evidence, prompt, model, or instructions.
- Do NOT reproduce internal reasoning.
- Start immediately with the answer to the question.
- If the information is not supported by the retrieved evidence, output exactly:
The provided documents do not contain this information.

For reasoning or scenario questions:
- Give the answer directly.
- Combine relevant information from the retrieved evidence when necessary.
- Do not describe the reasoning process.
- A short paragraph or a few concise bullet points are acceptable.

QUESTION:
{question}

RETRIEVED BANGLADESH BANK INFORMATION:
{context}

YOUR RESPONSE MUST CONTAIN ONLY THE ANSWER:
"""

In [ ]:
# ============================================================
# RETRIEVAL DEBUGGER
# ============================================================

def debug_retrieval(question, top_k=10):
    results = enhanced_retrieve(
        question,
        n_results=top_k,
        candidate_pool=20
    )

    print("\n" + "=" * 100)
    print("HYBRID RETRIEVAL DEBUG")
    print("=" * 100)
    print("Question:", question)

    if not results:
        print("No relevant source found.")
        return

    for rank, item in enumerate(results, start=1):
        meta = item.get("metadata", {})
        print("\n" + "-" * 100)
        print("Rank:", rank)
        print("PDF:", meta.get("pdf_name", ""))
        print("Type:", meta.get("document_type", ""))
        print("Year:", meta.get("year", ""))
        print("Period:", meta.get("reporting_period", ""))
        print("Section:", meta.get("section", ""))
        print("Page:", meta.get("page_start", ""))
        print("Dense score:", round(float(item.get("dense_score", 0.0)), 4))
        print("BM25 score:", round(float(item.get("bm25_score", 0.0)), 4))
        print("RRF score:", round(float(item.get("rrf_score", 0.0)), 6))
        print("Reranker score:", round(float(item.get("reranker_score", 0.0)), 4))
        print("Final score:", round(float(item.get("final_retrieval_score", 0.0)), 4))
        print("Text:", item.get("document", "")[:900].replace("\n", " "))


## 10. Hybrid retrieval + reranking

This section implements the methodology described in the project document: BM25 sparse retrieval + BGE dense retrieval, RRF fusion, metadata-aware scoring, and CrossEncoder reranking before answer generation.


In [ ]:
# ============================================================
# HYBRID RETRIEVAL + RERANKING + REASONING HELPERS
# ============================================================
# Pipeline:
#   Query expansion
#   -> Dense BGE retrieval
#   -> BM25 sparse retrieval
#   -> RRF fusion
#   -> lexical + metadata scoring
#   -> CrossEncoder reranking
#   -> final evidence selection
#
# Ground-truth data is NOT used during interactive retrieval.

import math
import re
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

STOPWORDS = {
    "what","was","were","is","are","the","a","an","of","to","in",
    "for","on","by","and","or","with","from","during","did","do",
    "does","how","why","which","that","this","their","its","as",
    "at","be","than","into","over","under","through","reported",
    "would","could","should","can","may","might","same","given",
    "tell","me","please","following","according"
}

RRF_K = 60
DENSE_CANDIDATES = 60
BM25_CANDIDATES = 60
RERANK_CANDIDATES = 12

_BM25_INDEX = None
_BM25_RECORDS = []
_RERANKER = None


def question_keywords(question):
    tokens = re.findall(
        r"[A-Za-z0-9][A-Za-z0-9%./_-]*",
        str(question).lower()
    )
    return {
        t for t in tokens
        if len(t) >= 2 and t not in STOPWORDS
    }


def lexical_match_score(question, document):
    q_terms = question_keywords(question)
    d_terms = question_keywords(document)
    if not q_terms:
        return 0.0
    return len(q_terms & d_terms) / len(q_terms)


def detect_question_type(question):
    q = str(question).lower()

    if any(x in q for x in [
        "if ", "suppose", "assume", "scenario", "eligible", "would be",
        "what happens", "how much can", "how much may", "under this case",
        "in this situation", "if an", "if a "
    ]):
        return "SCENARIO"

    if any(x in q for x in [
        "why ", "how did", "how does", "explain", "what factors",
        "what caused", "what led to", "reason for", "why was",
        "role of", "responsibility", "responsibilities", "function of",
        "functions of", "purpose of", "duties of"
    ]):
        return "REASONING"

    if any(x in q for x in [
        "percentage", "percent", "share", "ratio", "increase", "decrease",
        "growth", "change", "difference", "calculate", "how much"
    ]):
        return "CALCULATION"

    if any(x in q for x in [
        "compare", "compared with", "compared to", "higher than", "lower than",
        "versus", "vs.", "difference between"
    ]):
        return "COMPARISON"

    return "EXTRACTION"


def build_query_variants(question):
    q = str(question).strip()
    ql = q.lower()
    qtype = detect_question_type(q)

    variants = [q]

    # General semantic expansions.
    if qtype == "REASONING":
        variants.append(
            q + " role responsibilities functions duties purpose mandate"
        )
        variants.append(
            q + " committee responsibilities risk management oversight monitoring"
        )

    if qtype == "SCENARIO":
        variants.append(
            q + " rule condition limit threshold requirement applicable"
        )

    if qtype == "COMPARISON":
        variants.append(
            q + " compare values volume value share change"
        )

    if qtype == "CALCULATION":
        variants.append(
            q + " amount value volume percentage calculation"
        )

    # Domain-specific expansions.
    if "credit" in ql:
        variants.append(q + " credit risk management lending approval monitoring")
    if "committee" in ql:
        variants.append(q + " committee role responsibility function")
    if "loan" in ql:
        variants.append(q + " borrower credit loan monitoring risk")
    if "payment" in ql:
        variants.append(q + " payment transaction value volume platform")
    if "npl" in ql or "non-performing" in ql:
        variants.append(q + " non performing loan problem loan recovery")

    # Remove duplicates.
    out, seen = [], set()
    for v in variants:
        key = v.lower().strip()
        if key not in seen:
            seen.add(key)
            out.append(v)

    return out[:7]


def _tokenize_for_bm25(text):
    return re.findall(
        r"[A-Za-z0-9][A-Za-z0-9%./_-]*",
        str(text).lower()
    )


def get_bm25_index(force_rebuild=False):
    global _BM25_INDEX, _BM25_RECORDS

    if _BM25_INDEX is not None and not force_rebuild:
        return _BM25_INDEX, _BM25_RECORDS

    collection = get_collection()

    data = collection.get(
        where={"is_active": True},
        include=["documents", "metadatas"]
    )

    docs = data.get("documents") or []
    metas = data.get("metadatas") or []
    ids = data.get("ids") or []

    records = []
    corpus_tokens = []

    for idx, doc in enumerate(docs):
        doc = str(doc or "")
        if not doc.strip():
            continue

        meta = (
            metas[idx]
            if idx < len(metas) and metas[idx]
            else {}
        )

        rec_id = (
            ids[idx]
            if idx < len(ids)
            else str(idx)
        )

        records.append({
            "id": rec_id,
            "document": doc,
            "metadata": meta
        })

        corpus_tokens.append(
            _tokenize_for_bm25(doc)
        )

    if not corpus_tokens:
        _BM25_INDEX = None
        _BM25_RECORDS = []
        return None, []

    _BM25_INDEX = BM25Okapi(corpus_tokens)
    _BM25_RECORDS = records

    print(
        f"BM25 index ready: {len(records)} active chunks"
    )

    return _BM25_INDEX, _BM25_RECORDS


def get_reranker():
    global _RERANKER

    if _RERANKER is None:
        print(
            "Loading CrossEncoder reranker "
            "(first call downloads the model)..."
        )

        _RERANKER = CrossEncoder(
            "BAAI/bge-reranker-v2-m3",
            max_length=512
        )

    return _RERANKER


def _normalize_scores(values):
    if not values:
        return {}

    vals = [
        float(v)
        for v in values
    ]

    lo = min(vals)
    hi = max(vals)

    if hi - lo < 1e-12:
        return {
            i: 0.5
            for i in range(len(vals))
        }

    return {
        i: (v - lo) / (hi - lo)
        for i, v in enumerate(vals)
    }


def _metadata_bonus_normalized(question, meta):
    return float(
        _metadata_relevance_score(
            question,
            meta
        )
    )


def _dense_candidates(question, k=DENSE_CANDIDATES):
    model = get_embedder()
    collection = get_collection()

    total = collection.count()

    if total == 0:
        return []

    k = min(k, total)

    q_emb = model.encode(
        question,
        normalize_embeddings=True
    ).tolist()

    result = collection.query(
        query_embeddings=[q_emb],
        n_results=k,
        where={"is_active": True},
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    docs = result.get(
        "documents",
        [[]]
    )[0]

    metas = result.get(
        "metadatas",
        [[]]
    )[0]

    distances = result.get(
        "distances",
        [[]]
    )[0]

    out = []

    for rank, (doc, meta, dist) in enumerate(
        zip(docs, metas, distances),
        start=1
    ):
        out.append({
            "document": str(doc or ""),
            "metadata": meta or {},
            "dense_score": max(
                -1.0,
                min(
                    1.0,
                    1.0 - float(dist)
                )
            ),
            "dense_rank": rank
        })

    return out


def _bm25_candidates(question, k=BM25_CANDIDATES):
    bm25, records = get_bm25_index()

    if bm25 is None:
        return []

    tokens = _tokenize_for_bm25(question)

    if not tokens:
        return []

    scores = bm25.get_scores(tokens)

    order = sorted(
        range(len(scores)),
        key=lambda i: float(scores[i]),
        reverse=True
    )[:k]

    out = []

    for rank, idx in enumerate(
        order,
        start=1
    ):
        rec = records[idx]

        out.append({
            "id": rec["id"],
            "document": rec["document"],
            "metadata": rec["metadata"],
            "bm25_score": float(scores[idx]),
            "bm25_rank": rank
        })

    return out


def enhanced_retrieve(
    question,
    n_results=5,
    candidate_pool=60
):
    """
    Robust hybrid retrieval.

    Dense BGE and BM25 provide complementary candidates.
    RRF combines their rankings.
    Lexical overlap and metadata are soft signals.
    CrossEncoder is used as a reranker, not as the only decision maker.

    This is especially important for reasoning questions, where the correct
    evidence may use different wording from the question.
    """

    variants = build_query_variants(
        question
    )

    collected = {}

    # --------------------------------------------------------
    # Stage 1: Dense + BM25 over several query variants
    # --------------------------------------------------------

    for variant in variants:

        dense = _dense_candidates(
            variant,
            k=min(
                candidate_pool,
                DENSE_CANDIDATES
            )
        )

        sparse = _bm25_candidates(
            variant,
            k=min(
                candidate_pool,
                BM25_CANDIDATES
            )
        )

        for item in dense:

            key = re.sub(
                r"\s+",
                " ",
                item["document"].strip().lower()
            )

            entry = collected.setdefault(
                key,
                {
                    "document": item["document"],
                    "metadata": item["metadata"],
                    "dense_score": 0.0,
                    "bm25_score": 0.0,
                    "rrf_score": 0.0,
                    "variant_count": 0
                }
            )

            entry["dense_score"] = max(
                entry["dense_score"],
                item["dense_score"]
            )

            entry["rrf_score"] += (
                1.0
                / (RRF_K + item["dense_rank"])
            )

        for item in sparse:

            key = re.sub(
                r"\s+",
                " ",
                item["document"].strip().lower()
            )

            entry = collected.setdefault(
                key,
                {
                    "document": item["document"],
                    "metadata": item["metadata"],
                    "dense_score": 0.0,
                    "bm25_score": 0.0,
                    "rrf_score": 0.0,
                    "variant_count": 0
                }
            )

            entry["bm25_score"] = max(
                entry["bm25_score"],
                item["bm25_score"]
            )

            entry["rrf_score"] += (
                1.0
                / (RRF_K + item["bm25_rank"])
            )

    if not collected:
        return []

    candidates = list(
        collected.values()
    )

    # --------------------------------------------------------
    # Stage 2: calculate complementary signals
    # --------------------------------------------------------

    lexical_raw = [
        lexical_match_score(
            question,
            c["document"]
        )
        for c in candidates
    ]

    meta_raw = [
        _metadata_bonus_normalized(
            question,
            c["metadata"]
        )
        for c in candidates
    ]

    dense_raw = [
        c["dense_score"]
        for c in candidates
    ]

    bm25_raw = [
        c["bm25_score"]
        for c in candidates
    ]

    rrf_raw = [
        c["rrf_score"]
        for c in candidates
    ]

    lexical_norm = _normalize_scores(
        lexical_raw
    )

    meta_norm = _normalize_scores(
        meta_raw
    )

    dense_norm = _normalize_scores(
        dense_raw
    )

    bm25_norm = _normalize_scores(
        bm25_raw
    )

    rrf_norm = _normalize_scores(
        rrf_raw
    )

    for i, c in enumerate(candidates):

        c["lexical_score"] = float(
            lexical_raw[i]
        )

        c["metadata_bonus"] = float(
            meta_raw[i]
        )

        c["dense_score_normalized"] = float(
            dense_norm[i]
        )

        c["bm25_score_normalized"] = float(
            bm25_norm[i]
        )

        c["rrf_score_normalized"] = float(
            rrf_norm[i]
        )

        c["metadata_score"] = float(
            meta_norm[i]
        )

        # ----------------------------------------------------
        # Hybrid score before reranking
        #
        # Lexical overlap is deliberately included because
        # exact regulatory terminology is highly informative.
        # ----------------------------------------------------

        c["hybrid_score"] = (
            0.30 * c["rrf_score_normalized"]
            + 0.25 * c["dense_score_normalized"]
            + 0.25 * c["bm25_score_normalized"]
            + 0.15 * c["lexical_score"]
            + 0.05 * c["metadata_score"]
        )

    candidates.sort(
        key=lambda x: x["hybrid_score"],
        reverse=True
    )

    # Keep a broad pool for CrossEncoder.
    candidates = candidates[
        :min(
            len(candidates),
            RERANK_CANDIDATES
        )
    ]

    # --------------------------------------------------------
    # Stage 3: CrossEncoder reranking
    # --------------------------------------------------------

    reranker = get_reranker()

    pairs = [
        (
            question,
            c["document"]
        )
        for c in candidates
    ]

    rr_scores = reranker.predict(
        pairs,
        show_progress_bar=False
    )

    for c, rr in zip(
        candidates,
        rr_scores
    ):
        c["reranker_score"] = float(
            rr
        )

    rr_values = [
        c["reranker_score"]
        for c in candidates
    ]

    rr_norm = _normalize_scores(
        rr_values
    )

    # --------------------------------------------------------
    # Stage 4: Final score
    # --------------------------------------------------------
    #
    # Do NOT let the CrossEncoder completely override BM25 +
    # dense retrieval. This avoids generic-but-semantically-close
    # chunks winning over the exact regulatory terminology.
    # --------------------------------------------------------

    qtype = detect_question_type(
        question
    )

    for i, c in enumerate(
        candidates
    ):

        c["reranker_score_normalized"] = float(
            rr_norm[i]
        )

        if qtype == "REASONING":
            rerank_weight = 0.40
            hybrid_weight = 0.60
        else:
            rerank_weight = 0.50
            hybrid_weight = 0.50

        c["final_retrieval_score"] = (
            rerank_weight
            * c["reranker_score_normalized"]
            + hybrid_weight
            * c["hybrid_score"]
        )

    candidates.sort(
        key=lambda x: x["final_retrieval_score"],
        reverse=True
    )

    # --------------------------------------------------------
    # Stage 5: final evidence count
    # --------------------------------------------------------

    if qtype in {
        "REASONING",
        "SCENARIO",
        "COMPARISON",
        "CALCULATION"
    }:
        keep = min(
            n_results,
            6
        )
    else:
        keep = min(
            n_results,
            4
        )

    return candidates[:keep]


def compact_document_text(
    text,
    max_chars=3500
):
    text = str(text or "")

    if len(text) <= max_chars:
        return text

    return (
        text[:max_chars]
        + "\n[Evidence truncated for prompt size control]"
    )


def build_enhanced_context(
    question,
    evidence,
    max_sources=None,
    max_total_chars=18000
):
    qtype = detect_question_type(
        question
    )

    if max_sources is None:

        max_sources = (
            6
            if qtype in {
                "REASONING",
                "SCENARIO",
                "COMPARISON",
                "CALCULATION"
            }
            else 4
        )

    parts = []
    total = 0

    for rank, item in enumerate(
        evidence[:max_sources],
        start=1
    ):
        meta = item.get(
            "metadata",
            {}
        )

        content = compact_document_text(
            item.get(
                "document",
                ""
            ),
            3200
        )

        block = f"""
[Evidence {rank}]
PDF: {meta.get('pdf_name', 'Unknown')}
Document Type: {meta.get('document_type', 'Unknown')}
Year: {meta.get('year', '')}
Reporting Period: {meta.get('reporting_period', '')}
Circular Number: {meta.get('circular_number', '')}
Issue Date: {meta.get('issue_date', '')}
Chapter: {meta.get('chapter', '')}
Section: {meta.get('section', '')}
Page: {meta.get('page_start', 'N/A')}

Content:
{content}
"""

        if total + len(block) > max_total_chars:

            remaining = (
                max_total_chars
                - total
            )

            if remaining < 500:
                break

            block = block[:remaining]

        parts.append(block)
        total += len(block)

    return "\n".join(parts)


In [ ]:
def calculate_retrieval_confidence(evidence):

    if not evidence:
        return 0.0

    scores = []

    for item in evidence:

        score = float(
            item.get(
                "final_retrieval_score",
                0.0
            )
        )

        scores.append(score)

    if not scores:
        return 0.0

    # Top evidence score
    top_score = max(scores)

    # Convert to a bounded confidence score
    confidence = 1 / (1 + math.exp(-top_score))

    return round(
        confidence * 100,
        2
    )

In [ ]:
# ============================================================
# CELL 31: INTERACTIVE RAG QUESTION-ANSWERING
# ============================================================

import re
import time


# ============================================================
# 1. CLEAN LLM OUTPUT
# ============================================================

def clean_llm_answer(answer):

    if answer is None:
        return ""

    answer = str(answer).strip()

    if not answer:
        return ""

    # --------------------------------------------------------
    # Remove Qwen <think>...</think>
    # --------------------------------------------------------

    if re.search(r"</think>", answer, flags=re.IGNORECASE):

        answer = re.split(
            r"</think>",
            answer,
            maxsplit=1,
            flags=re.IGNORECASE
        )[1].strip()

    # Remove remaining complete think blocks
    answer = re.sub(
        r"(?is)<think>.*?</think>",
        "",
        answer
    )

    # Remove unclosed think block
    answer = re.sub(
        r"(?is)<think>.*$",
        "",
        answer
    )

    # --------------------------------------------------------
    # Remove answer labels
    # --------------------------------------------------------

    answer = re.sub(
        r"(?im)^\s*(?:final answer|final text|"
        r"direct answer|answer)\s*:\s*",
        "",
        answer
    )

    # --------------------------------------------------------
    # Remove internal reasoning lines
    # --------------------------------------------------------

    bad_patterns = [
        r"^\s*check\s*$",
        r"^\s*result\s*:",
        r"^\s*one minor tweak",
        r"^\s*one minor check",
        r"^\s*let'?s stick",
        r"^\s*let'?s double[- ]check",
        r"^\s*perfect\.?\s*$",
        r"^\s*proceed\.?\s*$",
        r"^\s*wait[,!]?.*$",
        r"^\s*end of thought process\.?\s*$",
        r"^\s*thinking process\s*:?",
        r"^\s*reasoning\s*:?",
        r"^\s*analysis\s*:?",
        r"^\s*self[- ]correction\s*:?",
        r"^\s*verification\s*:?",
        r"^\s*final check\s*:?",
        r"^\s*output generation\s*:?",
        r"^\s*final output generation\s*:?",
        r"^\s*i will\b",
        r"^\s*i'?ll\b",
        r"^\s*let me\b",
        r"^\s*we should\b",
        r"^\s*all constraints met",
        r"^\s*matches perfectly",
        r"^\s*matches evidence",
        r"^\s*matches exactly",
    ]

    cleaned_lines = []

    for line in answer.splitlines():

        line_clean = line.strip()

        if not line_clean:
            continue

        remove_line = False

        for pattern in bad_patterns:

            if re.search(
                pattern,
                line_clean,
                flags=re.IGNORECASE
            ):
                remove_line = True
                break

        if not remove_line:
            cleaned_lines.append(line)

    answer = "\n".join(cleaned_lines)

    # --------------------------------------------------------
    # Remove markdown fences
    # --------------------------------------------------------

    answer = re.sub(
        r"^\s*```(?:text|markdown)?\s*",
        "",
        answer,
        flags=re.IGNORECASE
    )

    answer = re.sub(
        r"\s*```\s*$",
        "",
        answer
    )

    # --------------------------------------------------------
    # Remove stray think tags
    # --------------------------------------------------------

    answer = re.sub(
        r"</?think>",
        "",
        answer,
        flags=re.IGNORECASE
    )

    # --------------------------------------------------------
    # Normalize blank lines
    # --------------------------------------------------------

    answer = re.sub(
        r"\n{2,}",
        "\n",
        answer
    ).strip()

    return answer


# ============================================================
# 2. SAFE LLM INVOCATION
# ============================================================

def invoke_llm_safely(
    prompt,
    max_retries=4
):

    for attempt in range(max_retries):

        try:

            return llm.invoke(prompt)

        except Exception as e:

            msg = str(e).lower()

            if (
                "429" not in msg
                and "rate limit" not in msg
            ):
                raise

            if attempt == max_retries - 1:
                raise

            time.sleep(
                2.0 * (attempt + 1)
            )

    raise RuntimeError(
        "LLM invocation failed."
    )


# ============================================================
# 3. RUN RAG ANSWER
# ============================================================

def run_rag_answer(question):

    qtype = detect_question_type(
        question
    )

    # --------------------------------------------------------
    # Retrieval settings
    # --------------------------------------------------------

    if qtype in {
        "REASONING",
        "SCENARIO",
        "COMPARISON",
        "CALCULATION"
    }:

        retrieval_k = 6
        candidate_pool = 80

    else:

        retrieval_k = 5
        candidate_pool = 80

    # --------------------------------------------------------
    # Hybrid retrieval + reranking
    # --------------------------------------------------------

    evidence = enhanced_retrieve(
        question,
        n_results=retrieval_k,
        candidate_pool=candidate_pool
    )
    confidence = calculate_retrieval_confidence(
    evidence
)

    # --------------------------------------------------------
    # No evidence
    # --------------------------------------------------------

    if not evidence:

        return {
            "answer":
                "The provided documents do not contain this information.",
            "evidence": [],
            "context": "",
            "question_type": qtype
        }

    # --------------------------------------------------------
    # Build context
    # --------------------------------------------------------

    context = build_enhanced_context(
        question,
        evidence,
        max_sources=retrieval_k,
        max_total_chars=18000
    )

    # --------------------------------------------------------
    # Build prompt
    # --------------------------------------------------------

    prompt = build_prompt(
        context,
        question
    )

    # --------------------------------------------------------
    # Call LLM
    # --------------------------------------------------------

    response = invoke_llm_safely(
        prompt
    )

    raw_answer = (
        response.content
        if hasattr(
            response,
            "content"
        )
        else str(response)
    )

    # --------------------------------------------------------
    # Clean answer
    # --------------------------------------------------------

    answer = clean_llm_answer(
        raw_answer
    )

    # --------------------------------------------------------
    # Final fallback
    # --------------------------------------------------------

    if not answer:

        answer = (
            "The provided documents do not contain this information."
        )

    return {
        "answer": answer,
        "evidence": evidence,
        "context": context,
        "question_type": qtype,
        "confidence": confidence
    }


# ============================================================
# 4. QUESTION INPUT
# ============================================================

while True:

    question = input(
        "\nEnter your question (type 'exit' to quit): "
    ).strip()

    if question.lower() == "exit":

        print(
            "Exiting RAG..."
        )

        break

    if not question:

        print(
            "Please enter a question."
        )

        continue

    print(
        "\n" + "=" * 80
    )

    print(
        "QUESTION"
    )

    print(
        "=" * 80
    )

    print(
        question
    )

    # --------------------------------------------------------
    # Run RAG
    # --------------------------------------------------------

    result = run_rag_answer(
        question
    )

    # --------------------------------------------------------
    # Answer
    # --------------------------------------------------------

    print(
        "\n" + "=" * 80
    )

    print(
        "ANSWER"
    )

    print(
        "=" * 80
    )

    print(
        result["answer"]
    )

    # --------------------------------------------------------
    # Source
    # --------------------------------------------------------

    print(
        "\n" + "=" * 80
    )

    print(
        "SOURCE"
    )

    print(
        "=" * 80
    )

    evidence = result.get(
        "evidence",
        []
    )

    if evidence:

        meta = (
            evidence[0].get(
                "metadata",
                {}
            )
            if isinstance(
                evidence[0],
                dict
            )
            else {}
        )

        print(
            "PDF          :",
            meta.get(
                "pdf_name",
                "Unknown"
            )
        )

        print(
            "Document Type:",
            meta.get(
                "document_type",
                "Unknown"
            )
        )

        for field, label in [
            (
                "circular_number",
                "Circular No."
            ),
            (
                "issue_date",
                "Issue Date"
            ),
            (
                "section",
                "Section"
            )
        ]:

            value = meta.get(
                field,
                ""
            )

            if value:

                print(
                    f"{label:13}: {value}"
                )

        page_start = meta.get(
            "page_start",
            ""
        )

        page_end = meta.get(
            "page_end",
            ""
        )

        if page_start:

            if (
                page_end
                and page_end != page_start
            ):

                print(
                    "Pages        :",
                    f"{page_start} - {page_end}"
                )

            else:

                print(
                    "Page         :",
                    page_start
                )

        print(
            "Question Type:",
            result.get(
                "question_type",
                ""
            )
        )

    else:

        print(
            "No relevant source found."
        )


Enter your question (type 'exit' to quit): exit
Exiting RAG...


In [ ]:
# ============================================================

# Complete Gradio Frontend — Google Colab
# ============================================================

!pip -q install -U gradio

import gradio as gr
import html
import os


# ============================================================
# IMPORTANT
# ============================================================
# These functions must already exist in your notebook:
#
# extract_text(pdf_path)
# extract_document_metadata(preview_text, pdf_path)
# ingest_pdf(pdf_path, doc_meta)
# run_rag_answer(question)
#
# Also make sure these variables exist:
#
# _BM25_INDEX
# _BM25_RECORDS
#
# If they already exist in your RAG code, DO NOT redefine them.


# ============================================================
# PROCESS MULTIPLE PDFs
# ============================================================

def process_uploaded_pdfs(pdf_paths):

    if not pdf_paths:
        return "⚠ Please upload at least one PDF."

    try:

        processed_count = 0

        for pdf_path in pdf_paths:

            # Extract text
            preview_text = extract_text(pdf_path)

            # Extract metadata
            doc_meta = extract_document_metadata(
                preview_text,
                pdf_path
            )

            # Add to RAG pipeline
            ingest_pdf(
                pdf_path,
                doc_meta
            )

            processed_count += 1


        # Reset BM25
        global _BM25_INDEX
        global _BM25_RECORDS

        _BM25_INDEX = None
        _BM25_RECORDS = []


        return (
            f"✓ Successfully processed "
            f"{processed_count} PDF(s). "
            f"Your regulatory knowledge base is ready."
        )


    except Exception as e:

        return (
            f"✕ Error processing PDF(s): {str(e)}"
        )


# ============================================================
# ASK RAG
# ============================================================

def ask_rag(question):

    # --------------------------------------------------------
    # Empty question
    # --------------------------------------------------------

    if not question or not question.strip():

        return (

            """
            <div class="empty-answer">

                <div class="empty-icon">
                    ?
                </div>

                <div>

                    <div class="empty-title">
                        No question entered
                    </div>

                    <div class="empty-text">
                        Please enter a regulatory question
                        to continue.
                    </div>

                </div>

            </div>
            """,

            """
            <div class="source-empty">
                No source available.
            </div>
            """
        )


    try:

        # ----------------------------------------------------
        # RUN RAG
        # ----------------------------------------------------

        result = run_rag_answer(question)


        # ----------------------------------------------------
        # Invalid result
        # ----------------------------------------------------

        if not isinstance(result, dict):

            answer_text = html.escape(
                str(result)
            )

            return (

                f"""
                <div class="answer-card">

                    <div class="answer-content">
                        {answer_text}
                    </div>

                </div>
                """,

                """
                <div class="source-empty">
                    No supporting source available.
                </div>
                """
            )


        # ----------------------------------------------------
        # ANSWER
        # ----------------------------------------------------

        answer = result.get(
            "answer",
            "No answer generated."
        )


        formatted_answer = html.escape(
            str(answer)
        ).replace(
            "\n",
            "<br>"
        )


        answer_html = f"""

        <div class="answer-card">

            <div class="answer-header">

                <div class="answer-title-wrapper">

                    <div class="answer-icon">
                        ✦
                    </div>

                    <div>

                        <div class="answer-title">
                            Regulatory Answer
                        </div>

                        <div class="answer-subtitle">
                            Generated from the uploaded
                            regulatory documents
                        </div>

                    </div>

                </div>

            </div>


            <div class="answer-divider"></div>


            <div class="answer-content">
                {formatted_answer}
            </div>

        </div>

        """


        # ----------------------------------------------------
        # EVIDENCE
        # ----------------------------------------------------

        evidence = result.get(
            "evidence",
            []
        )


        if not evidence:

            return (

                answer_html,

                """
                <div class="source-empty">

                    <span>
                        ⓘ
                    </span>

                    No supporting source found.

                </div>
                """
            )


        # ----------------------------------------------------
        # TOP EVIDENCE
        # ----------------------------------------------------

        top = evidence[0]


        metadata = (
            top.get("metadata", {})
            if isinstance(top, dict)
            else {}
        )


        # ----------------------------------------------------
        # SOURCE METADATA
        # ----------------------------------------------------

        pdf_name = metadata.get(
            "pdf_name",
            "Unknown document"
        )

        document_type = metadata.get(
            "document_type",
            "Not specified"
        )

        section = metadata.get(
            "section",
            "Not specified"
        )

        page_start = metadata.get(
            "page_start",
            ""
        )

        page_end = metadata.get(
            "page_end",
            ""
        )


        # ----------------------------------------------------
        # PAGE
        # ----------------------------------------------------

        if page_start:

            if (
                page_end
                and str(page_end) != str(page_start)
            ):

                page_text = (
                    f"Pages {page_start}–{page_end}"
                )

            else:

                page_text = (
                    f"Page {page_start}"
                )

        else:

            page_text = "Page unavailable"


        # ----------------------------------------------------
        # HTML ESCAPING
        # ----------------------------------------------------

        pdf_name = html.escape(
            str(pdf_name)
        )

        document_type = html.escape(
            str(document_type)
        )

        section = html.escape(
            str(section)
        )

        page_text = html.escape(
            str(page_text)
        )


        # ----------------------------------------------------
        # SOURCE CARD
        # ----------------------------------------------------

        source_html = f"""

        <div class="source-card">

            <div class="source-header">

                <div class="source-icon">
                    ▣
                </div>

                <div>

                    <div class="source-title">
                        Supporting Source
                    </div>

                    <div class="source-subtitle">
                        Highest-ranked evidence used
                        for this answer
                    </div>

                </div>

            </div>


            <div class="source-divider"></div>


            <div class="document-name">

                📄 {pdf_name}

            </div>


            <div class="metadata-grid">


                <div class="metadata-item">

                    <div class="metadata-label">
                        DOCUMENT TYPE
                    </div>

                    <div class="metadata-value">
                        {document_type}
                    </div>

                </div>


                <div class="metadata-item">

                    <div class="metadata-label">
                        SECTION
                    </div>

                    <div class="metadata-value">
                        {section}
                    </div>

                </div>


                <div class="metadata-item">

                    <div class="metadata-label">
                        LOCATION
                    </div>

                    <div class="metadata-value page-value">
                        {page_text}
                    </div>

                </div>


            </div>

        </div>

        """


        return answer_html, source_html


    except Exception as e:

        error_message = html.escape(
            str(e)
        )


        return (

            f"""

            <div class="error-card">

                <div class="error-icon">
                    !
                </div>

                <div>

                    <div class="error-title">
                        Unable to generate answer
                    </div>

                    <div class="error-text">
                        {error_message}
                    </div>

                </div>

            </div>

            """,

            """
            <div class="source-empty">
                Source unavailable.
            </div>
            """
        )


# ============================================================
# CUSTOM CSS
# ============================================================

custom_css = """

/* =========================================================
   COLORS
   ========================================================= */

:root {

    --green: #006A4E;

    --dark-green: #004D3A;

    --light-green: #EAF5F0;

    --gold: #C9A227;

    --light-gold: #F7F1DC;

    --background: #F5F8F6;

    --white: #FFFFFF;

    --text: #17231F;

    --muted: #68756F;

    --border: #DCE7E1;

    --danger: #B42318;

}


/* =========================================================
   MAIN APP
   ========================================================= */

.gradio-container {

    max-width: 1250px !important;

    margin: auto !important;

    background:

        radial-gradient(
            circle at 10% 10%,
            rgba(0,106,78,0.04),
            transparent 30%
        ),

        radial-gradient(
            circle at 90% 20%,
            rgba(201,162,39,0.04),
            transparent 30%
        ),

        var(--background) !important;

    font-family:

        Inter,
        ui-sans-serif,
        system-ui,
        -apple-system,
        BlinkMacSystemFont,
        "Segoe UI",
        sans-serif !important;

}


/* =========================================================
   HEADER
   ========================================================= */

.bb-header {

    position: relative;

    overflow: hidden;

    background:

        linear-gradient(
            135deg,
            #004D3A,
            #006A4E 60%,
            #08785A
        );

    border-radius: 22px;

    padding: 35px 40px;

    margin-bottom: 28px;

    color: white;

    box-shadow:
        0 15px 40px rgba(0,77,58,0.18);

}


.bb-header::before {

    content: "";

    position: absolute;

    width: 330px;

    height: 330px;

    right: -100px;

    top: -160px;

    border-radius: 50%;

    border:
        1px solid rgba(255,255,255,0.08);

    box-shadow:

        0 0 0 35px rgba(255,255,255,0.025),

        0 0 0 70px rgba(255,255,255,0.018);

}


.bb-header-content {

    position: relative;

    z-index: 2;

    display: flex;

    align-items: center;

    gap: 22px;

}


.bb-logo {

    width: 72px;

    height: 72px;

    background: white;

    border-radius: 18px;

    padding: 8px;

    object-fit: contain;

    box-shadow:
        0 8px 25px rgba(0,0,0,0.12);

}


.bb-title {

    margin: 0;

    font-size: 32px;

    font-weight: 750;

    letter-spacing: -0.8px;

}


.bb-subtitle {

    margin-top: 7px;

    font-size: 14px;

    color:
        rgba(255,255,255,0.82);

}


.bb-badge {

    display: inline-flex;

    margin-top: 13px;

    padding: 5px 11px;

    border-radius: 999px;

    background:
        rgba(255,255,255,0.11);

    border:
        1px solid rgba(255,255,255,0.14);

    font-size: 10px;

    font-weight: 700;

    letter-spacing: 0.5px;

}


/* =========================================================
   SECTION TITLES
   ========================================================= */

.section-title {

    font-size: 18px;

    font-weight: 750;

    color: var(--text);

    margin-bottom: 4px;

}


.section-description {

    color: var(--muted);

    font-size: 13px;

    margin-bottom: 15px;

}


/* =========================================================
   PANELS
   ========================================================= */

.panel {

    background: var(--white);

    border:
        1px solid var(--border);

    border-radius: 18px;

    padding: 22px;

    box-shadow:
        0 8px 30px rgba(0,60,45,0.07);

    margin-bottom: 20px;

}


/* =========================================================
   UPLOAD
   ========================================================= */

.file-upload {

    border:
        1.5px dashed #B9D2C8 !important;

    border-radius: 15px !important;

    background:
        linear-gradient(
            135deg,
            #F9FCFA,
            #F1F8F4
        ) !important;

    min-height: 155px !important;

}


.file-upload:hover {

    border-color:
        var(--green) !important;

    background:
        #F0F8F4 !important;

}


/* =========================================================
   BUTTONS
   ========================================================= */

button.primary {

    background:
        linear-gradient(
            135deg,
            var(--green),
            #08785A
        ) !important;

    border: none !important;

    color: white !important;

    font-weight: 700 !important;

    border-radius: 11px !important;

    min-height: 46px !important;

    box-shadow:
        0 7px 18px rgba(0,106,78,0.18) !important;

}


button.primary:hover {

    background:
        linear-gradient(
            135deg,
            var(--dark-green),
            var(--green)
        ) !important;

}


button.secondary {

    border:
        1px solid #B8D1C6 !important;

    background:
        white !important;

    color:
        var(--dark-green) !important;

    font-weight:
        650 !important;

    border-radius:
        11px !important;

}


/* =========================================================
   TEXT INPUT
   ========================================================= */

textarea,
input {

    border-radius:
        12px !important;

    border:
        1px solid var(--border) !important;

    background:
        white !important;

    color:
        var(--text) !important;

    box-shadow:
        none !important;

}


textarea:focus,
input:focus {

    border-color:
        var(--green) !important;

    box-shadow:
        0 0 0 3px rgba(0,106,78,0.08) !important;

}


/* =========================================================
   ANSWER CARD
   ========================================================= */

.answer-card {

    background:
        white;

    border:
        1px solid #CFE1D8;

    border-left:
        4px solid var(--green);

    border-radius:
        17px;

    padding:
        22px;

    box-shadow:
        0 10px 30px rgba(0,70,50,0.07);

    margin-bottom:
        18px;

}


.answer-header {

    display:
        flex;

    align-items:
        center;

}


.answer-title-wrapper {

    display:
        flex;

    align-items:
        center;

    gap:
        13px;

}


.answer-icon {

    width:
        39px;

    height:
        39px;

    display:
        flex;

    align-items:
        center;

    justify-content:
        center;

    border-radius:
        11px;

    background:
        var(--light-green);

    color:
        var(--green);

    font-size:
        19px;

    font-weight:
        800;

}


.answer-title {

    font-size:
        17px;

    font-weight:
        750;

    color:
        var(--text);

}


.answer-subtitle {

    font-size:
        11px;

    color:
        var(--muted);

    margin-top:
        3px;

}


.answer-divider {

    height:
        1px;

    background:
        var(--border);

    margin:
        18px 0;

}


.answer-content {

    color:
        #26342E;

    font-size:
        15px;

    line-height:
        1.85;

}


/* =========================================================
   SOURCE
   ========================================================= */

.source-card {

    background:
        #FBFDFC;

    border:
        1px solid var(--border);

    border-radius:
        17px;

    padding:
        21px;

    box-shadow:
        0 7px 22px rgba(0,60,45,0.045);

}


.source-header {

    display:
        flex;

    align-items:
        center;

    gap:
        13px;

}


.source-icon {

    width:
        39px;

    height:
        39px;

    display:
        flex;

    align-items:
        center;

    justify-content:
        center;

    border-radius:
        11px;

    background:
        var(--light-gold);

    color:
        #8A6C12;

    font-size:
        18px;

}


.source-title {

    font-size:
        16px;

    font-weight:
        750;

    color:
        var(--text);

}


.source-subtitle {

    font-size:
        11px;

    color:
        var(--muted);

    margin-top:
        3px;

}


.source-divider {

    height:
        1px;

    background:
        var(--border);

    margin:
        18px 0;

}


.document-name {

    padding:
        12px 14px;

    border-radius:
        10px;

    background:
        white;

    border:
        1px solid var(--border);

    color:
        var(--dark-green);

    font-weight:
        700;

    font-size:
        13px;

}


.metadata-grid {

    display:
        grid;

    grid-template-columns:
        repeat(3,1fr);

    gap:
        12px;

    margin-top:
        14px;

}


.metadata-item {

    padding:
        12px;

    background:
        white;

    border:
        1px solid var(--border);

    border-radius:
        10px;

}


.metadata-label {

    font-size:
        9px;

    font-weight:
        800;

    color:
        #87938D;

    letter-spacing:
        0.7px;

    margin-bottom:
        5px;

}


.metadata-value {

    font-size:
        12px;

    font-weight:
        650;

    color:
        var(--text);

}


.page-value {

    color:
        var(--green);

}


/* =========================================================
   EMPTY STATE
   ========================================================= */

.empty-answer {

    display:
        flex;

    align-items:
        center;

    gap:
        15px;

    padding:
        24px;

    background:
        white;

    border:
        1px solid var(--border);

    border-radius:
        16px;

}


.empty-icon {

    width:
        42px;

    height:
        42px;

    border-radius:
        12px;

    display:
        flex;

    align-items:
        center;

    justify-content:
        center;

    background:
        #F2F5F3;

    color:
        #75827C;

    font-weight:
        800;

}


.empty-title {

    font-weight:
        700;

    color:
        var(--text);

}


.empty-text {

    font-size:
        12px;

    color:
        var(--muted);

    margin-top:
        3px;

}


.source-empty {

    padding:
        20px;

    border:
        1px dashed #CDDbd4;

    border-radius:
        14px;

    background:
        #FAFCFB;

    color:
        var(--muted);

    text-align:
        center;

    font-size:
        13px;

}


/* =========================================================
   ERROR
   ========================================================= */

.error-card {

    display:
        flex;

    gap:
        14px;

    padding:
        20px;

    background:
        #FFF7F6;

    border:
        1px solid #F0C9C5;

    border-left:
        4px solid var(--danger);

    border-radius:
        15px;

}


.error-icon {

    width:
        38px;

    height:
        38px;

    display:
        flex;

    align-items:
        center;

    justify-content:
        center;

    border-radius:
        10px;

    background:
        #FDE9E7;

    color:
        var(--danger);

    font-weight:
        800;

}


.error-title {

    font-weight:
        750;

    color:
        #8E2118;

}


.error-text {

    margin-top:
        4px;

    color:
        #8E2118;

    font-size:
        12px;

}


/* =========================================================
   FOOTER
   ========================================================= */

.bb-footer {

    text-align:
        center;

    padding:
        20px 0 8px;

    color:
        #84918B;

    font-size:
        11px;

}


.bb-footer strong {

    color:
        var(--green);

}


/* =========================================================
   MOBILE
   ========================================================= */

@media (max-width: 800px) {

    .bb-header {

        padding:
            25px;

    }

    .bb-header-content {

        align-items:
            flex-start;

    }

    .bb-logo {

        width:
            58px;

        height:
            58px;

    }

    .bb-title {

        font-size:
            25px;

    }

    .metadata-grid {

        grid-template-columns:
            1fr;

    }

}
"""


# ============================================================
# GRADIO APP
# ============================================================

with gr.Blocks(

    title="BanglaRegu-RAG",

    css=custom_css,

    theme=gr.themes.Soft(

        primary_hue="green",

        neutral_hue="slate"

    )

) as demo:


    # ========================================================
    # HEADER
    # ========================================================

    gr.HTML(

        """

        <div class="bb-header">

            <div class="bb-header-content">


                <img

                    class="bb-logo"

                    src="https://www.bb.org.bd/smespd_portal/res/icons/logo-bb.svg"

                    alt="Bangladesh Bank"

                >


                <div>

                    <h1 class="bb-title">
                        RegAssist
                    </h1>


                    <div class="bb-subtitle">

                        Bangladesh Bank Regulatory
                        Question Answering Assistant

                    </div>


                    <div class="bb-badge">

                        AI-POWERED REGULATORY
                        KNOWLEDGE SYSTEM

                    </div>

                </div>


            </div>

        </div>

        """

    )


    # ========================================================
    # MAIN
    # ========================================================

    with gr.Row(equal_height=False):


        # ====================================================
        # LEFT
        # ====================================================

        with gr.Column(scale=5):


            # ------------------------------------------------
            # DOCUMENTS
            # ------------------------------------------------

            gr.HTML(

                """

                <div class="section-title">

                    Regulatory Knowledge Base

                </div>


                <div class="section-description">

                    Upload Bangladesh Bank circulars,
                    guidelines, notices or regulatory PDFs.

                </div>

                """

            )


            with gr.Group(
                elem_classes=["panel"]
            ):


                pdf = gr.File(

                    label="Upload PDF documents",

                    file_types=[".pdf"],

                    type="filepath",

                    file_count="multiple",

                    elem_classes=["file-upload"]

                )


                process = gr.Button(

                    "Process Documents",

                    variant="primary"

                )


                status = gr.Textbox(

                    label="Processing Status",

                    lines=2,

                    interactive=False

                )


            # ------------------------------------------------
            # QUESTION
            # ------------------------------------------------

            gr.HTML(

                """

                <div class="section-title">

                    Ask the Regulatory Assistant

                </div>


                <div class="section-description">

                    Ask questions based on the documents
                    available in the knowledge base.

                </div>

                """

            )


            with gr.Group(
                elem_classes=["panel"]
            ):


                question = gr.Textbox(

                    label="Your question",

                    placeholder=(

                        "Example: What are the reporting "
                        "requirements under this circular?"

                    ),

                    lines=5

                )


                with gr.Row():


                    clear = gr.ClearButton(

                        components=[question],

                        value="Clear"

                    )


                    submit = gr.Button(

                        "Ask Question",

                        variant="primary"

                    )


        # ====================================================
        # RIGHT
        # ====================================================

        with gr.Column(scale=7):


            gr.HTML(

                """

                <div class="section-title">

                    Regulatory Response

                </div>


                <div class="section-description">

                    AI-generated response with supporting
                    regulatory evidence.

                </div>

                """

            )


            # ------------------------------------------------
            # ANSWER
            # ------------------------------------------------

            answer = gr.HTML(

                value="""

                <div class="empty-answer">

                    <div class="empty-icon">
                        ✦
                    </div>


                    <div>

                        <div class="empty-title">

                            Ready for your question

                        </div>


                        <div class="empty-text">

                            Upload documents and ask a
                            regulatory question to begin.

                        </div>

                    </div>

                </div>

                """

            )


            # ------------------------------------------------
            # SOURCE
            # ------------------------------------------------

            gr.HTML(

                """

                <div
                    class="section-title"
                    style="margin-top:20px;"
                >

                    Supporting Evidence

                </div>

                """

            )


            source = gr.HTML(

                value="""

                <div class="source-empty">

                    Source information will appear here
                    after a question is answered.

                </div>

                """

            )


    # ========================================================
    # FOOTER
    # ========================================================

    gr.HTML(

        """

        <div class="bb-footer">

            <strong>BanglaRegu-RAG</strong>

            &nbsp;•&nbsp;

            Regulatory Question Answering System

            <br>

            Document-grounded regulatory
            information retrieval.

        </div>

        """

    )


    # ========================================================
    # EVENTS
    # ========================================================

    process.click(

        fn=process_uploaded_pdfs,

        inputs=pdf,

        outputs=status

    )


    submit.click(

        fn=ask_rag,

        inputs=question,

        outputs=[
            answer,
            source
        ]

    )


# ============================================================
# LAUNCH FOR GOOGLE COLAB
# ============================================================

demo.launch(

    share=True,

    debug=True

)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 6.7 MB/s eta 0:00:00


/tmp/ipykernel_2830/1854570277.py:1530: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://fe6acf2bef3fab6548.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://fe6acf2bef3fab6548.gradio.live


In [ ]:
# ============================================================
# FINAL ANSWER EVALUATION
# SEMANTIC ANSWER SIMILARITY (%) + F1 SCORE (%)
# ============================================================
#
# INPUT COLUMNS:
#   Question
#   Ground_Truth_answer
#   Model Answer
#   PDF_Name
#   Page
#
# OUTPUT COLUMNS:
#   Same original 5 columns
#   +
#   Answer Similarity (%)
#   Answer F1 (%)
#
# IMPORTANT:
#   NO RAG GENERATION
#   NO API CALL
#   NO API KEY
#   NO GEMINI TOKEN USAGE
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import re
from google.colab import files


# ============================================================
# 2. UPLOAD FINAL EXCEL FILE
# ============================================================

print("=" * 70)
print("UPLOAD FINAL EXCEL FILE")
print("=" * 70)

uploaded = files.upload()

input_file = list(uploaded.keys())[0]

df = pd.read_excel(input_file)

print("\n✓ File loaded successfully.")
print("Input file:", input_file)
print("Total questions:", len(df))


# ============================================================
# 3. CHECK REQUIRED 5 COLUMNS
# ============================================================

required_columns = [
    "Question",
    "Ground_Truth_answer",
    "Model Answer",
    "PDF_Name",
    "Page"
]

for column in required_columns:

    if column not in df.columns:

        raise ValueError(
            f"ERROR: '{column}' column is missing."
        )


print("\n✓ All 5 required columns found.")


# ============================================================
# 4. CHECK FOR MISSING ANSWERS
# ============================================================

def is_blank(value):

    if value is None:
        return True

    if pd.isna(value):
        return True

    return str(value).strip() == ""


missing_indices = []

for idx in df.index:

    ground_truth = df.loc[
        idx,
        "Ground_Truth_answer"
    ]

    model_answer = df.loc[
        idx,
        "Model Answer"
    ]

    if (
        is_blank(ground_truth)
        or
        is_blank(model_answer)
    ):

        missing_indices.append(idx)


print("\n" + "=" * 70)
print("ANSWER STATUS")
print("=" * 70)

print(
    "Total questions      :",
    len(df)
)

print(
    "Complete answer pairs:",
    len(df) - len(missing_indices)
)

print(
    "Missing answer pairs :",
    len(missing_indices)
)


# ============================================================
# 5. STOP IF ANY ANSWER IS MISSING
# ============================================================

if missing_indices:

    print(
        "\n⚠️ Evaluation stopped."
    )

    print(
        "Missing question numbers:"
    )

    print(
        [
            idx + 1
            for idx in missing_indices
        ]
    )

    raise ValueError(
        "Every question must have both "
        "Ground_Truth_answer and Model Answer."
    )


# ============================================================
# 6. INSTALL / IMPORT SENTENCE TRANSFORMERS
# ============================================================

try:

    from sentence_transformers import (
        SentenceTransformer,
        util
    )

except ImportError:

    print(
        "\nInstalling sentence-transformers..."
    )

    !pip -q install sentence-transformers

    from sentence_transformers import (
        SentenceTransformer,
        util
    )


# ============================================================
# 7. LOAD LOCAL SEMANTIC SIMILARITY MODEL
# ============================================================

print("\n" + "=" * 70)
print("LOADING SEMANTIC SIMILARITY MODEL")
print("=" * 70)

similarity_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print(
    "✓ Similarity model loaded."
)


# ============================================================
# 8. PREPARE GROUND TRUTH + MODEL ANSWERS
# ============================================================

ground_truth_answers = (
    df["Ground_Truth_answer"]
    .astype(str)
    .str.strip()
    .tolist()
)

model_answers = (
    df["Model Answer"]
    .astype(str)
    .str.strip()
    .tolist()
)


# ============================================================
# 9. CREATE EMBEDDINGS
# ============================================================

print("\n" + "=" * 70)
print("CREATING ANSWER EMBEDDINGS")
print("=" * 70)

ground_truth_embeddings = (
    similarity_model.encode(
        ground_truth_answers,
        convert_to_tensor=True,
        show_progress_bar=True
    )
)

model_answer_embeddings = (
    similarity_model.encode(
        model_answers,
        convert_to_tensor=True,
        show_progress_bar=True
    )
)

print(
    "\n✓ Embeddings created."
)


# ============================================================
# 10. CALCULATE COSINE SIMILARITY
# ============================================================

print("\n" + "=" * 70)
print("CALCULATING ANSWER SIMILARITY")
print("=" * 70)

similarity_matrix = util.cos_sim(
    ground_truth_embeddings,
    model_answer_embeddings
)


# ============================================================
# 11. EXTRACT MATCHING PAIR SIMILARITY
# ============================================================

similarity_scores = []

for i in range(len(df)):

    score = similarity_matrix[
        i,
        i
    ].item()

    percentage = score * 100

    # Keep between 0 and 100
    percentage = max(
        0,
        min(
            100,
            percentage
        )
    )

    similarity_scores.append(
        percentage
    )


# ============================================================
# 12. TOKENIZATION FOR F1
# ============================================================

def tokenize(text):

    text = str(text).lower()

    return re.findall(
        r"\b\w+\b",
        text
    )


# ============================================================
# 13. F1 CALCULATION FUNCTION
# ============================================================

def calculate_f1(
    reference,
    prediction
):

    reference_tokens = tokenize(
        reference
    )

    prediction_tokens = tokenize(
        prediction
    )

    if (
        len(reference_tokens) == 0
        or
        len(prediction_tokens) == 0
    ):

        return 0.0


    # --------------------------------------------------------
    # Count reference tokens
    # --------------------------------------------------------

    reference_counts = {}

    for token in reference_tokens:

        reference_counts[token] = (
            reference_counts.get(token, 0) + 1
        )


    # --------------------------------------------------------
    # Count prediction tokens
    # --------------------------------------------------------

    prediction_counts = {}

    for token in prediction_tokens:

        prediction_counts[token] = (
            prediction_counts.get(token, 0) + 1
        )


    # --------------------------------------------------------
    # Count overlapping tokens
    # --------------------------------------------------------

    overlap = 0

    for token in prediction_counts:

        if token in reference_counts:

            overlap += min(
                prediction_counts[token],
                reference_counts[token]
            )


    if overlap == 0:

        return 0.0


    # --------------------------------------------------------
    # Precision
    # --------------------------------------------------------

    precision = (
        overlap
        /
        len(prediction_tokens)
    )


    # --------------------------------------------------------
    # Recall
    # --------------------------------------------------------

    recall = (
        overlap
        /
        len(reference_tokens)
    )


    # --------------------------------------------------------
    # F1
    # --------------------------------------------------------

    if precision + recall == 0:

        return 0.0


    f1 = (
        2
        *
        precision
        *
        recall
        /
        (precision + recall)
    )

    return f1


# ============================================================
# 14. CALCULATE F1 FOR ALL QUESTIONS
# ============================================================

print("\n" + "=" * 70)
print("CALCULATING TOKEN-LEVEL F1")
print("=" * 70)

f1_scores = []

for idx in df.index:

    reference = df.loc[
        idx,
        "Ground_Truth_answer"
    ]

    prediction = df.loc[
        idx,
        "Model Answer"
    ]

    f1 = calculate_f1(
        reference,
        prediction
    )

    f1_percentage = f1 * 100

    f1_scores.append(
        f1_percentage
    )


# ============================================================
# 15. ADD BOTH METRICS
# ============================================================

df[
    "Answer Similarity (%)"
] = similarity_scores

df[
    "Answer F1 (%)"
] = f1_scores


# ============================================================
# 16. DISPLAY INDIVIDUAL RESULTS
# ============================================================

print("\n" + "=" * 70)
print("INDIVIDUAL RESULTS")
print("=" * 70)

for i in range(len(df)):

    print(
        f"Q{i + 1:<4} | "
        f"Similarity: "
        f"{similarity_scores[i]:.2f}% | "
        f"F1: "
        f"{f1_scores[i]:.2f}%"
    )


# ============================================================
# 17. CALCULATE AVERAGES
# ============================================================

average_similarity = (
    sum(similarity_scores)
    /
    len(similarity_scores)
)

average_f1 = (
    sum(f1_scores)
    /
    len(f1_scores)
)


# ============================================================
# 18. FINAL RESULTS
# ============================================================

print("\n" + "=" * 70)
print("FINAL EVALUATION RESULT")
print("=" * 70)

print(
    "\nTotal questions evaluated:",
    len(df)
)

print(
    f"Average Answer Similarity: "
    f"{average_similarity:.2f}%"
)

print(
    f"Average Token-level F1: "
    f"{average_f1:.2f}%"
)


# ============================================================
# 19. SAVE FINAL EXCEL
# ============================================================

final_file = (
    "rag_final_similarity_f1_evaluation.xlsx"
)

df.to_excel(
    final_file,
    index=False
)


print(
    "\n✓ Original columns preserved:"
)

print(
    [
        "Question",
        "Ground_Truth_answer",
        "Model Answer",
        "PDF_Name",
        "Page"
    ]
)

print(
    "\n✓ Added:"
)

print(
    [
        "Answer Similarity (%)",
        "Answer F1 (%)"
    ]
)

print(
    "\n✓ Final file saved as:"
)

print(
    final_file
)


# ============================================================
# 20. DOWNLOAD FINAL FILE
# ============================================================

files.download(
    final_file
)

print(
    "\n✓ Evaluation completed successfully."
)

UPLOAD FINAL EXCEL FILE


KeyboardInterrupt: 

In [ ]:
# ============================================================
# ANSWER-LEVEL PRECISION & RECALL EVALUATION
# ============================================================
#
# INPUT COLUMNS:
#   Question
#   Ground_Truth_answer
#   Model Answer
#   PDF_Name
#   Page
#
# OUTPUT:
#   Original 5 columns +
#   Answer Precision (%)
#   Answer Recall (%)
#
# NO RAG MODEL
# NO API CALL
# NO TOKEN USAGE
# ============================================================


import pandas as pd
import re
from google.colab import files


# ============================================================
# 1. UPLOAD FINAL EXCEL FILE
# ============================================================

print("=" * 70)
print("UPLOAD FINAL EXCEL FILE")
print("=" * 70)

uploaded = files.upload()

input_file = list(uploaded.keys())[0]

df = pd.read_excel(input_file)

print("\n✓ File loaded successfully.")
print("Total questions:", len(df))


# ============================================================
# 2. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    "Question",
    "Ground_Truth_answer",
    "Model Answer",
    "PDF_Name",
    "Page"
]

for column in required_columns:

    if column not in df.columns:

        raise ValueError(
            f"ERROR: '{column}' column is missing."
        )

print("\n✓ All 5 required columns found.")


# ============================================================
# 3. CHECK FOR MISSING ANSWERS
# ============================================================

def is_blank(value):

    if value is None:
        return True

    if pd.isna(value):
        return True

    return str(value).strip() == ""


missing_indices = []

for idx in df.index:

    ground_truth = df.loc[
        idx,
        "Ground_Truth_answer"
    ]

    model_answer = df.loc[
        idx,
        "Model Answer"
    ]

    if (
        is_blank(ground_truth)
        or
        is_blank(model_answer)
    ):

        missing_indices.append(idx)


print("\n" + "=" * 70)
print("ANSWER STATUS")
print("=" * 70)

print(
    "Total questions      :",
    len(df)
)

print(
    "Complete answer pairs:",
    len(df) - len(missing_indices)
)

print(
    "Missing answer pairs :",
    len(missing_indices)
)


if missing_indices:

    print(
        "\nMissing question numbers:"
    )

    print(
        [idx + 1 for idx in missing_indices]
    )

    raise ValueError(
        "Some questions are missing Ground_Truth_answer "
        "or Model Answer."
    )


# ============================================================
# 4. TOKENIZATION
# ============================================================

def tokenize(text):

    text = str(text).lower()

    # Extract words and numbers
    tokens = re.findall(
        r"\b\w+\b",
        text
    )

    return tokens


# ============================================================
# 5. PRECISION & RECALL FUNCTION
# ============================================================

def calculate_precision_recall(
    reference,
    prediction
):

    reference_tokens = tokenize(
        reference
    )

    prediction_tokens = tokenize(
        prediction
    )


    # --------------------------------------------------------
    # Handle empty answers
    # --------------------------------------------------------

    if len(prediction_tokens) == 0:

        precision = 0.0

    else:

        precision = None


    if len(reference_tokens) == 0:

        recall = 0.0

    else:

        recall = None


    if (
        len(reference_tokens) == 0
        or
        len(prediction_tokens) == 0
    ):

        return 0.0, 0.0


    # --------------------------------------------------------
    # Count reference tokens
    # --------------------------------------------------------

    reference_counts = {}

    for token in reference_tokens:

        reference_counts[token] = (
            reference_counts.get(token, 0) + 1
        )


    # --------------------------------------------------------
    # Count prediction tokens
    # --------------------------------------------------------

    prediction_counts = {}

    for token in prediction_tokens:

        prediction_counts[token] = (
            prediction_counts.get(token, 0) + 1
        )


    # --------------------------------------------------------
    # Calculate overlapping tokens
    # --------------------------------------------------------

    overlap = 0

    for token in prediction_counts:

        if token in reference_counts:

            overlap += min(
                prediction_counts[token],
                reference_counts[token]
            )


    # --------------------------------------------------------
    # Precision
    #
    # How much of the model answer overlaps
    # with the ground-truth answer?
    # --------------------------------------------------------

    precision = (
        overlap
        /
        len(prediction_tokens)
    )


    # --------------------------------------------------------
    # Recall
    #
    # How much of the ground-truth answer
    # is covered by the model answer?
    # --------------------------------------------------------

    recall = (
        overlap
        /
        len(reference_tokens)
    )


    return precision, recall


# ============================================================
# 6. CALCULATE PRECISION & RECALL
# ============================================================

print("\n" + "=" * 70)
print("CALCULATING ANSWER PRECISION & RECALL")
print("=" * 70)


precision_scores = []
recall_scores = []


for idx in df.index:

    reference = df.loc[
        idx,
        "Ground_Truth_answer"
    ]

    prediction = df.loc[
        idx,
        "Model Answer"
    ]


    precision, recall = (
        calculate_precision_recall(
            reference,
            prediction
        )
    )


    precision_percentage = (
        precision * 100
    )

    recall_percentage = (
        recall * 100
    )


    precision_scores.append(
        precision_percentage
    )

    recall_scores.append(
        recall_percentage
    )


# ============================================================
# 7. ADD RESULTS TO DATASET
# ============================================================

df[
    "Answer Precision (%)"
] = precision_scores

df[
    "Answer Recall (%)"
] = recall_scores


# ============================================================
# 8. DISPLAY INDIVIDUAL RESULTS
# ============================================================

print("\n" + "=" * 70)
print("INDIVIDUAL PRECISION & RECALL")
print("=" * 70)


for i in range(len(df)):

    print(
        f"Q{i + 1:<4} | "
        f"Precision: "
        f"{precision_scores[i]:.2f}% | "
        f"Recall: "
        f"{recall_scores[i]:.2f}%"
    )


# ============================================================
# 9. CALCULATE AVERAGES
# ============================================================

average_precision = (
    sum(precision_scores)
    /
    len(precision_scores)
)

average_recall = (
    sum(recall_scores)
    /
    len(recall_scores)
)


# ============================================================
# 10. FINAL RESULTS
# ============================================================

print("\n" + "=" * 70)
print("FINAL PRECISION & RECALL RESULT")
print("=" * 70)


print(
    "\nTotal questions evaluated:",
    len(df)
)

print(
    f"Average Answer Precision: "
    f"{average_precision:.2f}%"
)

print(
    f"Average Answer Recall: "
    f"{average_recall:.2f}%"
)


# ============================================================
# 11. SAVE FINAL EXCEL
# ============================================================

final_file = (
    "rag_answer_precision_recall_results.xlsx"
)

df.to_excel(
    final_file,
    index=False
)


print(
    "\n✓ Original 5 columns preserved."
)

print(
    "✓ Precision column added."
)

print(
    "✓ Recall column added."
)

print(
    "\n✓ Final file saved as:"
)

print(
    final_file
)


# ============================================================
# 12. DOWNLOAD
# ============================================================

files.download(
    final_file
)

print(
    "\n✓ Evaluation completed successfully."
)

In [ ]:
# ============================================================
# NUMERIC ANSWER ACCURACY / MATCH
# ============================================================
#
# Required columns:
#   Question
#   Ground_Truth_answer
#   Model Answer
#   PDF_Name
#   Page
#
# No RAG model
# No API
# No token usage
# ============================================================

import pandas as pd
import re
from google.colab import files


# ============================================================
# 1. UPLOAD FINAL EXCEL FILE
# ============================================================

print("=" * 70)
print("UPLOAD FINAL EXCEL FILE")
print("=" * 70)

uploaded = files.upload()

input_file = list(uploaded.keys())[0]

df = pd.read_excel(input_file)

print("\n✓ File loaded.")
print("Total questions:", len(df))


# ============================================================
# 2. CHECK COLUMNS
# ============================================================

required_columns = [
    "Question",
    "Ground_Truth_answer",
    "Model Answer",
    "PDF_Name",
    "Page"
]

for col in required_columns:
    if col not in df.columns:
        raise ValueError(
            f"ERROR: '{col}' column is missing."
        )


# ============================================================
# 3. EXTRACT NUMBERS
# ============================================================

def extract_numbers(text):

    text = str(text)

    # Remove commas inside numbers:
    # 1,250,000 -> 1250000
    text = re.sub(
        r'(?<=\d),(?=\d)',
        '',
        text
    )

    # Extract:
    # percentages: 42.5%
    # decimals: 12.50
    # integers: 100
    # negative numbers: -25
    numbers = re.findall(
        r'-?\d+(?:\.\d+)?%?',
        text
    )

    return numbers


# ============================================================
# 4. NORMALIZE NUMBERS
# ============================================================

def normalize_number(number):

    number = str(number).strip()

    is_percentage = "%" in number

    number = number.replace(
        "%",
        ""
    )

    try:
        value = float(number)

        # Normalize 50.0 -> 50
        value = round(
            value,
            6
        )

        return (
            value,
            is_percentage
        )

    except:

        return (
            None,
            is_percentage
        )


# ============================================================
# 5. CALCULATE NUMERIC MATCH
# ============================================================

def numeric_match(
    reference,
    prediction
):

    reference_numbers = extract_numbers(
        reference
    )

    prediction_numbers = extract_numbers(
        prediction
    )

    reference_normalized = [
        normalize_number(x)
        for x in reference_numbers
    ]

    prediction_normalized = [
        normalize_number(x)
        for x in prediction_numbers
    ]

    reference_normalized = [
        x for x in reference_normalized
        if x[0] is not None
    ]

    prediction_normalized = [
        x for x in prediction_normalized
        if x[0] is not None
    ]


    # --------------------------------------------------------
    # No numeric information in ground truth
    # --------------------------------------------------------

    if len(reference_normalized) == 0:

        return None


    # --------------------------------------------------------
    # No numbers generated by model
    # --------------------------------------------------------

    if len(prediction_normalized) == 0:

        return 0.0


    # --------------------------------------------------------
    # Count matched numbers
    # --------------------------------------------------------

    matched = 0

    used_prediction = set()


    for ref_value, ref_percent in reference_normalized:

        for j, (
            pred_value,
            pred_percent
        ) in enumerate(
            prediction_normalized
        ):

            if j in used_prediction:
                continue

            # Percentage/non-percentage should match
            if ref_percent != pred_percent:
                continue

            # Exact numeric match
            if ref_value == pred_value:

                matched += 1

                used_prediction.add(j)

                break


    # --------------------------------------------------------
    # Recall-style numeric accuracy
    #
    # How many ground-truth numbers were correctly captured?
    # --------------------------------------------------------

    score = (
        matched
        /
        len(reference_normalized)
    )

    return score * 100


# ============================================================
# 6. CALCULATE SCORES
# ============================================================

numeric_scores = []

for idx in df.index:

    reference = df.loc[
        idx,
        "Ground_Truth_answer"
    ]

    prediction = df.loc[
        idx,
        "Model Answer"
    ]

    score = numeric_match(
        reference,
        prediction
    )

    numeric_scores.append(
        score
    )


# ============================================================
# 7. ADD RESULT
# ============================================================

df[
    "Numeric Accuracy (%)"
] = numeric_scores


# ============================================================
# 8. DISPLAY RESULTS
# ============================================================

print("\n" + "=" * 70)
print("NUMERIC ACCURACY RESULTS")
print("=" * 70)

for i, score in enumerate(
    numeric_scores
):

    if score is None:

        print(
            f"Q{i+1:<4}: No numeric information"
        )

    else:

        print(
            f"Q{i+1:<4}: {score:.2f}%"
        )


# ============================================================
# 9. AVERAGE
# ============================================================

valid_scores = [
    x for x in numeric_scores
    if x is not None
]

if len(valid_scores) > 0:

    average_numeric_accuracy = (
        sum(valid_scores)
        /
        len(valid_scores)
    )

else:

    average_numeric_accuracy = 0


# ============================================================
# 10. FINAL RESULT
# ============================================================

print("\n" + "=" * 70)
print("FINAL NUMERIC ACCURACY")
print("=" * 70)

print(
    f"\nAverage Numeric Accuracy: "
    f"{average_numeric_accuracy:.2f}%"
)

print(
    "Questions with numeric answers:",
    len(valid_scores)
)


# ============================================================
# 11. SAVE RESULT
# ============================================================

output_file = (
    "rag_numeric_accuracy_results.xlsx"
)

df.to_excel(
    output_file,
    index=False
)

print(
    "\n✓ Numeric accuracy saved."
)

print(
    "✓ Output:",
    output_file
)


# ============================================================
# 12. DOWNLOAD
# ============================================================

files.download(
    output_file
)

In [ ]:
# ============================================================
# BATCH + CHECKPOINT RAG ANSWER GENERATION
# ============================================================

import pandas as pd
import os
import time
from google.colab import files

# ------------------------------------------------------------
# 1. Upload dataset
# ------------------------------------------------------------

uploaded = files.upload()

input_file = list(uploaded.keys())[0]

df = pd.read_excel(input_file)

print("Dataset loaded.")
print("Total questions:", len(df))


# ------------------------------------------------------------
# 2. Required columns
# ------------------------------------------------------------

if "Question" not in df.columns:
    raise ValueError("Column 'Question' is missing.")

if "Model Answer" not in df.columns:
    df["Model Answer"] = ""


# ------------------------------------------------------------
# 3. Checkpoint file
# ------------------------------------------------------------

checkpoint_file = "rag_evaluation_checkpoint.xlsx"

# If checkpoint already exists, use it
if os.path.exists(checkpoint_file):

    print("\nExisting checkpoint found.")
    print("Loading previous progress...")

    old_df = pd.read_excel(
        checkpoint_file
    )

    # Make sure Question column exists
    if "Question" not in old_df.columns:
        raise ValueError(
            "Checkpoint does not contain 'Question'."
        )

    # Copy previous Model Answers
    if "Model Answer" in old_df.columns:

        answer_map = dict(
            zip(
                old_df["Question"].astype(str).str.strip(),
                old_df["Model Answer"]
            )
        )

        for i in df.index:

            q = str(
                df.loc[i, "Question"]
            ).strip()

            if q in answer_map:

                previous_answer = answer_map[q]

                if (
                    pd.notna(previous_answer)
                    and str(previous_answer).strip()
                ):

                    df.loc[i, "Model Answer"] = (
                        previous_answer
                    )

    print("Previous answers restored.")


# ------------------------------------------------------------
# 4. Function to detect blank answers
# ------------------------------------------------------------

def is_blank(value):

    if pd.isna(value):
        return True

    return str(value).strip() == ""


# ------------------------------------------------------------
# 5. Find unanswered questions
# ------------------------------------------------------------

remaining = [
    i
    for i in df.index
    if is_blank(
        df.loc[i, "Model Answer"]
    )
]


print("\n" + "=" * 70)
print("CHECKPOINT STATUS")
print("=" * 70)

print("Total questions :", len(df))
print(
    "Already answered:",
    len(df) - len(remaining)
)
print(
    "Remaining       :",
    len(remaining)
)


# ------------------------------------------------------------
# 6. Batch processing
# ------------------------------------------------------------

BATCH_SIZE = 10

total_remaining = len(remaining)

for batch_start in range(
    0,
    total_remaining,
    BATCH_SIZE
):

    batch_indices = remaining[
        batch_start:
        batch_start + BATCH_SIZE
    ]

    print("\n" + "=" * 70)

    print(
        f"BATCH "
        f"{batch_start // BATCH_SIZE + 1}"
    )

    print(
        f"Questions "
        f"{batch_start + 1}"
        f"-"
        f"{min(batch_start + BATCH_SIZE, total_remaining)}"
        f" of {total_remaining}"
    )

    print("=" * 70)


    # --------------------------------------------------------
    # Process questions in current batch
    # --------------------------------------------------------

    for idx in batch_indices:

        question = str(
            df.loc[idx, "Question"]
        ).strip()

        print(
            f"\nProcessing Q{idx + 1}:"
        )

        print(question[:150])


        try:

            result = run_rag_answer(
                question
            )

            # -----------------------------------------------
            # Extract answer
            # -----------------------------------------------

            if isinstance(
                result,
                dict
            ):

                answer = result.get(
                    "answer",
                    ""
                )

            else:

                answer = str(result)


            # -----------------------------------------------
            # Save successful answer
            # -----------------------------------------------

            if (
                answer is not None
                and str(answer).strip()
            ):

                df.loc[
                    idx,
                    "Model Answer"
                ] = str(answer).strip()

                print(
                    "✓ Answer generated."
                )

            else:

                print(
                    "✗ Empty answer."
                )


        except Exception as e:

            print(
                "✗ Error:",
                str(e)
            )

            # Leave blank and continue
            continue


        # ----------------------------------------------------
        # IMPORTANT:
        # Save checkpoint after EVERY question
        # ----------------------------------------------------

        df.to_excel(
            checkpoint_file,
            index=False
        )

        print(
            "✓ Checkpoint saved."
        )

        # Small delay
        time.sleep(2)


    # --------------------------------------------------------
    # Save again after each batch
    # --------------------------------------------------------

    df.to_excel(
        checkpoint_file,
        index=False
    )

    print(
        "\n✓ Batch checkpoint saved."
    )


# ------------------------------------------------------------
# 7. Final status
# ------------------------------------------------------------

remaining_after = [
    i
    for i in df.index
    if is_blank(
        df.loc[i, "Model Answer"]
    )
]


print("\n" + "=" * 70)
print("FINAL CHECKPOINT STATUS")
print("=" * 70)

print(
    "Total questions     :",
    len(df)
)

print(
    "Answers available   :",
    len(df) - len(remaining_after)
)

print(
    "Answers still blank :",
    len(remaining_after)
)

if remaining_after:

    print(
        "\nStill unanswered:"
    )

    print(
        [
            i + 1
            for i in remaining_after
        ]
    )

else:

    print(
        "\n✓ All answers generated successfully."
    )


# ------------------------------------------------------------
# 8. Download checkpoint
# ------------------------------------------------------------

files.download(
    checkpoint_file
)

In [ ]:
# ============================================================
# PART 2
# FINAL ANSWER SIMILARITY EVALUATION
# ============================================================

import pandas as pd
from sentence_transformers import SentenceTransformer, util
from google.colab import files


# ============================================================
# 1. LOAD COMPLETED CHECKPOINT
# ============================================================
# ============================================================
# 1. UPLOAD COMPLETED CHECKPOINT FILE
# ============================================================



print("=" * 70)
print("UPLOAD COMPLETED CHECKPOINT FILE")
print("=" * 70)

uploaded = files.upload()

input_file = list(uploaded.keys())[0]

df = pd.read_excel(input_file)

print("\n✓ File loaded successfully.")
print("Input file:", input_file)
print("Total questions:", len(df))


# ============================================================
# 2. CHECK REQUIRED COLUMNS
# ============================================================

required_columns = [
    "Question",
    "Answer",
    "Model Answer"
]

for column in required_columns:

    if column not in df.columns:

        raise ValueError(
            f"ERROR: '{column}' column is missing."
        )


# ============================================================
# 3. CHECK THAT ALL 102 ANSWERS EXIST
# ============================================================

def is_blank(value):

    if value is None:
        return True

    if pd.isna(value):
        return True

    return str(value).strip() == ""


blank_indices = [
    idx
    for idx in df.index
    if is_blank(
        df.loc[idx, "Model Answer"]
    )
]


print("\n" + "=" * 70)
print("ANSWER STATUS")
print("=" * 70)

print(
    "Total questions :",
    len(df)
)

print(
    "Completed answers:",
    len(df) - len(blank_indices)
)

print(
    "Blank answers    :",
    len(blank_indices)
)


# ============================================================
# STOP IF ANY ANSWER IS MISSING
# ============================================================

if len(blank_indices) > 0:

    print(
        "\n⚠️ Similarity calculation NOT started."
    )

    print(
        "Please complete all answers first."
    )

    print(
        "Missing question numbers:"
    )

    print(
        [
            idx + 1
            for idx in blank_indices
        ]
    )

else:

    # ========================================================
    # 4. LOAD SEMANTIC SIMILARITY MODEL
    # ========================================================

    print("\n" + "=" * 70)
    print("LOADING SIMILARITY MODEL")
    print("=" * 70)

    similarity_model = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2"
    )

    print(
        "✓ Similarity model loaded."
    )


    # ========================================================
    # 5. PREPARE GOLD + MODEL ANSWERS
    # ========================================================

    reference_answers = (
        df["Answer"]
        .astype(str)
        .str.strip()
        .tolist()
    )

    model_answers = (
        df["Model Answer"]
        .astype(str)
        .str.strip()
        .tolist()
    )


    # ========================================================
    # 6. CREATE EMBEDDINGS
    # ========================================================

    print(
        "\nCreating answer embeddings..."
    )

    reference_embeddings = (
        similarity_model.encode(
            reference_answers,
            convert_to_tensor=True,
            show_progress_bar=True
        )
    )

    model_embeddings = (
        similarity_model.encode(
            model_answers,
            convert_to_tensor=True,
            show_progress_bar=True
        )
    )


    # ========================================================
    # 7. CALCULATE COSINE SIMILARITY
    # ========================================================

    similarity_matrix = util.cos_sim(
        reference_embeddings,
        model_embeddings
    )


    # ========================================================
    # 8. GET MATCHING ANSWER SCORES
    # ========================================================

    similarity_scores = []

    for i in range(
        len(df)
    ):

        score = similarity_matrix[
            i,
            i
        ].item()

        percentage = score * 100

        # Keep score between 0 and 100
        percentage = max(
            0,
            min(
                100,
                percentage
            )
        )

        similarity_scores.append(
            percentage
        )


    # ========================================================
    # 9. ADD SCORE TO DATASET
    # ========================================================

    df[
        "Answer Similarity (%)"
    ] = similarity_scores


    # ========================================================
    # 10. DISPLAY INDIVIDUAL RESULTS
    # ========================================================

    print("\n" + "=" * 70)
    print("INDIVIDUAL ANSWER SIMILARITY")
    print("=" * 70)

    for i, score in enumerate(
        similarity_scores
    ):

        print(
            f"Q{i + 1:<4}: "
            f"{score:.2f}%"
        )


    # ========================================================
    # 11. CALCULATE AVERAGE
    # ========================================================

    average_similarity = (
        sum(similarity_scores)
        /
        len(similarity_scores)
    )


    # ========================================================
    # 12. FINAL RESULT
    # ========================================================

    print("\n" + "=" * 70)
    print("FINAL EVALUATION RESULT")
    print("=" * 70)

    print(
        "\nQuestions evaluated:",
        len(similarity_scores)
    )

    print(
        f"Average Answer Similarity: "
        f"{average_similarity:.2f}%"
    )


    # ========================================================
    # 13. SAVE FINAL EXCEL
    # ========================================================

    final_file = (
        "rag_final_evaluation.xlsx"
    )

    df.to_excel(
        final_file,
        index=False
    )

    print(
        "\n✓ Similarity scores saved."
    )

    print(
        "✓ Final file:",
        final_file
    )

    print(
        "\n✓ Evaluation completed successfully."
    )

### User-facing Bangladesh Bank RAG chatbot

Type questions directly in the interactive cell above. The system supports extraction, comparison, calculation, scenario-based, and reasoning questions using the uploaded Bangladesh Bank PDFs. Ground-truth Excel is used only in the separate evaluation section.


In [ ]:
# ============================================================
# CELL 28: UPLOAD GROUND-TRUTH DATASET
# ============================================================

import pandas as pd
import numpy as np
import re
import os

# ------------------------------------------------------------
# Google Colab upload
# ------------------------------------------------------------

try:
    from google.colab import files

    print("Please upload your ground-truth Excel file...")

    uploaded = files.upload()

    if len(uploaded) == 0:
        raise ValueError("No file was uploaded.")

    ground_truth_file = list(uploaded.keys())[0]

except ImportError:
    # If running outside Google Colab
    ground_truth_file = input(
        "Enter the path of your ground-truth Excel file: "
    )

print("\nGround-truth file:")
print(ground_truth_file)


# ------------------------------------------------------------
# Read Excel
# ------------------------------------------------------------

ground_truth_df = pd.read_excel(
    ground_truth_file
)

print("\nDataset loaded successfully.")
print("Number of questions:", len(ground_truth_df))

print("\nColumns:")
print(ground_truth_df.columns.tolist())

display(ground_truth_df.head())

In [ ]:
# ============================================================
# CELL 29: PREPARE GROUND-TRUTH DATASET
# ============================================================

ground_truth_df = ground_truth_df.dropna(how="all").reset_index(drop=True)

def find_column(df, possible_names):
    normalized = {str(col).strip().lower().replace(" ", "_"): col for col in df.columns}
    for name in possible_names:
        key = name.strip().lower().replace(" ", "_")
        if key in normalized:
            return normalized[key]
    return None

QUESTION_COL = find_column(ground_truth_df, ["question"])
ANSWER_COL = find_column(ground_truth_df, ["answer"])
PDF_COL = find_column(ground_truth_df, ["pdf", "pdf_name", "document"])
PAGE_COL = find_column(ground_truth_df, ["page", "page_number"])
SECTION_COL = find_column(ground_truth_df, ["section"])

required = {
    "Question": QUESTION_COL,
    "Answer": ANSWER_COL,
    "PDF": PDF_COL,
    "Page": PAGE_COL,
    "Section": SECTION_COL,
}

print("Detected columns:")
for label, col in required.items():
    print(f"{label:10}: {col}")

missing = [label for label, col in required.items() if col is None]
if missing:
    raise ValueError(
        "Missing required ground-truth columns: " + ", ".join(missing) +
        ". Expected columns: Question, Answer, PDF, Page, Section"
    )

print("\nDataset validation successful.")


In [ ]:
# ============================================================
# CELL 30: TEXT NORMALIZATION
# ============================================================

def normalize_text(text):

    if text is None:
        return ""

    try:
        if pd.isna(text):
            return ""
    except Exception:
        pass

    text = str(text).lower().strip()

    # Normalize unicode dashes
    text = text.replace("–", "-")
    text = text.replace("—", "-")
    text = text.replace("-", "-")

    # Normalize PDF/path separators
    text = text.replace("\\", "/")

    # Normalize whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

In [ ]:
# ============================================================
# CELL 36: RETRIEVAL EVALUATION
#
# Dataset:
# Question, Answer, PDF, Page, Section
#
# Evidence column is NOT used.
# ============================================================

import re
import pandas as pd


# ============================================================
# 1. PDF NORMALIZATION
# ============================================================

def _norm_pdf(name):

    if name is None:
        return ""

    try:
        if pd.isna(name):
            return ""
    except Exception:
        pass

    s = str(name).strip().lower()

    # Remove path
    s = s.replace("\\", "/")
    s = s.split("/")[-1]

    # Remove .pdf
    s = re.sub(
        r"\.pdf$",
        "",
        s,
        flags=re.IGNORECASE
    )

    # Remove duplicate suffix
    s = re.sub(
        r"\(\d+\)$",
        "",
        s
    )

    # Normalize punctuation
    s = re.sub(
        r"[^a-z0-9]+",
        " ",
        s
    )

    return s.strip()


# ============================================================
# 2. PAGE PARSER
# ============================================================

def _page_interval(value):

    if value is None:
        return None

    try:
        if pd.isna(value):
            return None
    except Exception:
        pass

    s = str(value).strip()

    if not s:
        return None

    nums = [
        int(x)
        for x in re.findall(
            r"\d+",
            s
        )
    ]

    if not nums:
        return None

    if len(nums) == 1:

        return (
            nums[0],
            nums[0]
        )

    return (
        min(nums),
        max(nums)
    )


# ============================================================
# 3. SECTION MATCH
# ============================================================

def _section_match(
    retrieved_section,
    ground_truth_section
):

    r = normalize_text(
        retrieved_section
    )

    g = normalize_text(
        ground_truth_section
    )

    if not r or not g:
        return False

    return (
        r == g
        or r in g
        or g in r
    )


# ============================================================
# 4. SOURCE RELEVANCE
# ============================================================

def evidence_match(
    retrieved_meta,
    ground_truth_pdf,
    ground_truth_page,
    ground_truth_section
):

    meta = (
        retrieved_meta
        if isinstance(
            retrieved_meta,
            dict
        )
        else {}
    )

    # --------------------------------------------------------
    # PDF
    # --------------------------------------------------------

    r_pdf = _norm_pdf(
        meta.get(
            "pdf_name",
            ""
        )
    )

    g_pdf = _norm_pdf(
        ground_truth_pdf
    )

    if not r_pdf or not g_pdf:
        return False

    pdf_match = (
        r_pdf == g_pdf
        or r_pdf in g_pdf
        or g_pdf in r_pdf
    )

    if not pdf_match:
        return False

    # --------------------------------------------------------
    # Section
    # --------------------------------------------------------

    section_match = _section_match(
        meta.get(
            "section",
            ""
        ),
        ground_truth_section
    )

    # --------------------------------------------------------
    # Ground-truth page
    # --------------------------------------------------------

    gt_page = _page_interval(
        ground_truth_page
    )

    # --------------------------------------------------------
    # Retrieved page range
    # --------------------------------------------------------

    retrieved_start = _page_interval(
        meta.get(
            "page_start",
            ""
        )
    )

    retrieved_end = _page_interval(
        meta.get(
            "page_end",
            ""
        )
    )

    page_match = False

    if gt_page and retrieved_start:

        gt_lo, gt_hi = gt_page

        r_lo, _ = retrieved_start

        if retrieved_end:

            _, r_hi = retrieved_end

        else:

            r_hi = r_lo

        # Direct overlap
        page_match = not (
            r_hi < gt_lo
            or r_lo > gt_hi
        )

        # ----------------------------------------------------
        # Allow nearby pages.
        #
        # This is important because a structure-aware chunk
        # can contain text spanning adjacent pages.
        # ----------------------------------------------------

        if not page_match:

            distance = min(
                abs(r_hi - gt_lo),
                abs(r_lo - gt_hi)
            )

            if distance <= 1:

                page_match = True

    # ========================================================
    # RELEVANCE RULE
    # ========================================================

    # Same PDF + page
    if pdf_match and page_match:
        return True

    # Same PDF + section
    if pdf_match and section_match:
        return True

    # If page/section unavailable,
    # same PDF is acceptable.
    if (
        pdf_match
        and not str(
            ground_truth_page
        ).strip()
        and not str(
            ground_truth_section
        ).strip()
    ):
        return True

    return False


# ============================================================
# 5. RETRIEVAL EVALUATION
# ============================================================

def evaluate_retrieval(
    df,
    top_k=10
):

    required_columns = [
        QUESTION_COL,
        ANSWER_COL,
        PDF_COL,
        PAGE_COL,
        SECTION_COL
    ]

    missing = [
        col
        for col in required_columns
        if col not in df.columns
    ]

    if missing:

        raise ValueError(
            "Missing required columns: "
            + str(missing)
        )

    rows = []

    for row_number, (_, row) in enumerate(
        df.iterrows(),
        start=1
    ):

        question = str(
            row[QUESTION_COL]
        ).strip()

        gt_answer = str(
            row[ANSWER_COL]
        ).strip()

        gt_pdf = (
            str(
                row[PDF_COL]
            ).strip()
            if pd.notna(
                row[PDF_COL]
            )
            else ""
        )

        gt_page = (
            str(
                row[PAGE_COL]
            ).strip()
            if pd.notna(
                row[PAGE_COL]
            )
            else ""
        )

        gt_section = (
            str(
                row[SECTION_COL]
            ).strip()
            if pd.notna(
                row[SECTION_COL]
            )
            else ""
        )

        # ----------------------------------------------------
        # Retrieve MORE candidates than final top-k
        # ----------------------------------------------------

        try:

            results = enhanced_retrieve(
                question,
                n_results=top_k,
                candidate_pool=80
            )

        except Exception as e:

            print(
                f"Retrieval error for "
                f"question {row_number}: {e}"
            )

            results = []

        relevance = []

        retrieved_pdfs = []
        retrieved_pages = []
        retrieved_sections = []

        # ----------------------------------------------------
        # Evaluate top-k
        # ----------------------------------------------------

        for item in results[:top_k]:

            meta = (
                item.get(
                    "metadata",
                    {}
                )
                if isinstance(
                    item,
                    dict
                )
                else {}
            )

            if not isinstance(
                meta,
                dict
            ):
                meta = {}

            retrieved_pdfs.append(
                meta.get(
                    "pdf_name",
                    ""
                )
            )

            retrieved_pages.append(
                meta.get(
                    "page_start",
                    ""
                )
            )

            retrieved_sections.append(
                meta.get(
                    "section",
                    ""
                )
            )

            relevant = evidence_match(
                retrieved_meta=meta,
                ground_truth_pdf=gt_pdf,
                ground_truth_page=gt_page,
                ground_truth_section=gt_section
            )

            relevance.append(
                1 if relevant else 0
            )

        # ----------------------------------------------------
        # Pad
        # ----------------------------------------------------

        while len(relevance) < top_k:

            relevance.append(0)

            retrieved_pdfs.append("")

            retrieved_pages.append("")

            retrieved_sections.append("")

        # ----------------------------------------------------
        # Save
        # ----------------------------------------------------

        rows.append({

            "question":
                question,

            "ground_truth_answer":
                gt_answer,

            "ground_truth_pdf":
                gt_pdf,

            "ground_truth_page":
                gt_page,

            "ground_truth_section":
                gt_section,

            "relevance":
                relevance,

            "retrieved_pdfs":
                retrieved_pdfs,

            "retrieved_pages":
                retrieved_pages,

            "retrieved_sections":
                retrieved_sections
        })

        print(
            f"Evaluated "
            f"{row_number}/{len(df)}"
        )

    return pd.DataFrame(
        rows
    )


# ============================================================
# RUN
# ============================================================

retrieval_results_df = evaluate_retrieval(
    ground_truth_df,
    top_k=10
)

print(
    "=" * 70
)

print(
    "RETRIEVAL EVALUATION DATA"
)

print(
    "=" * 70
)

print(
    "Questions evaluated:",
    len(
        retrieval_results_df
    )
)

display(
    retrieval_results_df.head(10)
)

In [ ]:
# ============================================================
# CELL 32: RUN RETRIEVAL EVALUATION
# ============================================================

def evaluate_retrieval(
    df,
    top_k=10
):

    retrieval_results = []

    # ========================================================
    # CHECK DATASET COLUMNS
    # ========================================================

    required_columns = [
        QUESTION_COL,
        ANSWER_COL,
        PDF_COL,
        PAGE_COL,
        SECTION_COL
    ]

    missing_columns = [
        col
        for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:

        raise ValueError(
            f"Missing required columns: "
            f"{missing_columns}"
        )

    # ========================================================
    # LOOP THROUGH DATASET
    # ========================================================

    for row_number, (_, row) in enumerate(
        df.iterrows(),
        start=1
    ):

        # ----------------------------------------------------
        # Ground truth question
        # ----------------------------------------------------

        question = str(
            row[QUESTION_COL]
        ).strip()

        # ----------------------------------------------------
        # Ground truth answer
        # ----------------------------------------------------

        gt_answer = str(
            row[ANSWER_COL]
        ).strip()

        # ----------------------------------------------------
        # Ground truth PDF
        # ----------------------------------------------------

        if pd.notna(row[PDF_COL]):

            gt_pdf = str(
                row[PDF_COL]
            ).strip()

        else:

            gt_pdf = ""

        # ----------------------------------------------------
        # Ground truth Page
        # ----------------------------------------------------

        if pd.notna(row[PAGE_COL]):

            gt_page = str(
                row[PAGE_COL]
            ).strip()

        else:

            gt_page = ""

        # ----------------------------------------------------
        # Ground truth Section
        # ----------------------------------------------------

        if pd.notna(row[SECTION_COL]):

            gt_section = str(
                row[SECTION_COL]
            ).strip()

        else:

            gt_section = ""

        # ====================================================
        # RETRIEVE
        # ====================================================

        try:

            evidence = enhanced_retrieve(
                question,
                n_results=top_k,
                candidate_pool=max(
                    50,
                    top_k
                )
            )

        except Exception as e:

            print(
                f"Retrieval error "
                f"for question {row_number}: {e}"
            )

            evidence = []

        # ====================================================
        # CALCULATE RELEVANCE
        # ====================================================

        relevance = []

        retrieved_pdfs = []
        retrieved_pages = []
        retrieved_sections = []

        for item in evidence[:top_k]:

            # ------------------------------------------------
            # Make sure item is a dictionary
            # ------------------------------------------------

            if not isinstance(
                item,
                dict
            ):

                item = {}

            # ------------------------------------------------
            # Metadata
            # ------------------------------------------------

            retrieved_meta = item.get(
                "metadata",
                {}
            )

            if not isinstance(
                retrieved_meta,
                dict
            ):

                retrieved_meta = {}

            # ------------------------------------------------
            # Store retrieved metadata
            # ------------------------------------------------

            retrieved_pdfs.append(
                retrieved_meta.get(
                    "pdf_name",
                    ""
                )
            )

            retrieved_pages.append(
                retrieved_meta.get(
                    "page_start",
                    ""
                )
            )

            retrieved_sections.append(
                retrieved_meta.get(
                    "section",
                    ""
                )
            )

            # ------------------------------------------------
            # IMPORTANT:
            # Use ONLY the new evidence_match signature
            # ------------------------------------------------

            is_relevant = evidence_match(

                retrieved_meta=retrieved_meta,

                ground_truth_pdf=gt_pdf,

                ground_truth_page=gt_page,

                ground_truth_section=gt_section
            )

            relevance.append(
                1 if is_relevant else 0
            )

        # ====================================================
        # PAD RESULTS TO TOP-K
        # ====================================================

        while len(relevance) < top_k:

            relevance.append(0)

            retrieved_pdfs.append("")

            retrieved_pages.append("")

            retrieved_sections.append("")

        # ====================================================
        # SAVE RESULT
        # ====================================================

        retrieval_results.append({

            "question":
                question,

            "ground_truth_answer":
                gt_answer,

            "ground_truth_pdf":
                gt_pdf,

            "ground_truth_page":
                gt_page,

            "ground_truth_section":
                gt_section,

            "relevance":
                relevance,

            "retrieved_pdfs":
                retrieved_pdfs,

            "retrieved_pages":
                retrieved_pages,

            "retrieved_sections":
                retrieved_sections
        })

        # ====================================================
        # PROGRESS
        # ====================================================

        print(
            f"Evaluated "
            f"{row_number}/{len(df)}"
        )

    # ========================================================
    # CREATE DATAFRAME
    # ========================================================

    return pd.DataFrame(
        retrieval_results
    )


# ============================================================
# RUN RETRIEVAL EVALUATION
# ============================================================

retrieval_results_df = evaluate_retrieval(
    ground_truth_df,
    top_k=10
)


# ============================================================
# DISPLAY
# ============================================================

print("=" * 70)
print("RETRIEVAL EVALUATION DATA")
print("=" * 70)

print(
    "Questions evaluated:",
    len(retrieval_results_df)
)

display(
    retrieval_results_df.head(10)
)

In [ ]:
# ============================================================
# CELL 33: RETRIEVAL RESULTS SUMMARY
# ============================================================

import math
import pandas as pd


# ============================================================
# Metric functions
# ============================================================

def recall_at_k(relevance, k):

    relevance = relevance[:k]

    return 1.0 if any(
        r == 1
        for r in relevance
    ) else 0.0


def precision_at_k(relevance, k):

    relevance = relevance[:k]

    if not relevance:
        return 0.0

    return sum(relevance) / len(relevance)


def reciprocal_rank(relevance):

    for rank, relevance_value in enumerate(
        relevance,
        start=1
    ):

        if relevance_value == 1:

            return 1.0 / rank

    return 0.0


def ndcg_at_k(relevance, k):

    relevance = relevance[:k]

    if not relevance:
        return 0.0

    # DCG
    dcg = 0.0

    for i, rel in enumerate(
        relevance
    ):

        dcg += (
            (2 ** rel - 1)
            / math.log2(i + 2)
        )

    # Ideal DCG
    ideal = sorted(
        relevance,
        reverse=True
    )

    idcg = 0.0

    for i, rel in enumerate(
        ideal
    ):

        idcg += (
            (2 ** rel - 1)
            / math.log2(i + 2)
        )

    if idcg == 0:

        return 0.0

    return dcg / idcg


# ============================================================
# Calculate metrics for every question
# ============================================================

metric_rows = []


for _, row in retrieval_results_df.iterrows():

    relevance = row["relevance"]

    # --------------------------------------------------------
    # Convert string list to actual list if necessary
    # --------------------------------------------------------

    if isinstance(
        relevance,
        str
    ):

        try:

            relevance = eval(
                relevance
            )

        except Exception:

            relevance = []

    relevance = list(
        relevance
    )

    # --------------------------------------------------------
    # Calculate metrics
    # --------------------------------------------------------

    metric_rows.append({

        "Question":
            row["question"],

        "Recall@1":
            recall_at_k(
                relevance,
                1
            ),

        "Recall@5":
            recall_at_k(
                relevance,
                5
            ),

        "Recall@10":
            recall_at_k(
                relevance,
                10
            ),

        "Precision@5":
            precision_at_k(
                relevance,
                5
            ),

        "Precision@10":
            precision_at_k(
                relevance,
                10
            ),

        "MRR":
            reciprocal_rank(
                relevance
            ),

        "NDCG@10":
            ndcg_at_k(
                relevance,
                10
            )
    })


# ============================================================
# Create metric dataframe
# ============================================================

retrieval_metrics_df = pd.DataFrame(
    metric_rows
)


# ============================================================
# Calculate overall average
# ============================================================

retrieval_metric_columns = [

    "Recall@1",
    "Recall@5",
    "Recall@10",
    "Precision@5",
    "Precision@10",
    "MRR",
    "NDCG@10"

]


retrieval_summary = (
    retrieval_metrics_df[
        retrieval_metric_columns
    ]
    .mean()
)


# ============================================================
# Display overall results
# ============================================================

print(
    "=========================================="
)

print(
    "       RETRIEVAL EVALUATION RESULTS"
)

print(
    "=========================================="
)


for metric, value in (
    retrieval_summary.items()
):

    print(
        f"{metric:15s}: "
        f"{value:.4f}"
    )


print(
    "=========================================="
)


# ============================================================
# Display per-question results
# ============================================================

print(
    "\nPER-QUESTION RETRIEVAL METRICS"
)

display(
    retrieval_metrics_df.head(10)
)

## 13. Answer evaluation

This benchmark compares the RAG-generated answer with the Ground Truth Answer. It does not replace the interactive chatbot. The Excel is used only when running this experiment.


In [ ]:
# ============================================================
# ANSWER + CONTEXT EVALUATION
# Metrics:
#   Token F1
#   Answer Similarity
#   Context Similarity
#
# Exact Match is intentionally removed.
# ============================================================

from difflib import SequenceMatcher
import time
from collections import Counter
import numpy as np


def normalized_answer(text):
    text = normalize_text(text)

    text = re.sub(
        r"[^a-z0-9%./-]+",
        " ",
        text
    )

    return re.sub(
        r"\s+",
        " ",
        text
    ).strip()


def token_f1_score(pred, truth):
    p = normalized_answer(pred).split()
    t = normalized_answer(truth).split()

    if not p or not t:
        return 0.0

    cp = Counter(p)
    ct = Counter(t)

    overlap = sum(
        (cp & ct).values()
    )

    if overlap == 0:
        return 0.0

    precision = (
        overlap / len(p)
    )

    recall = (
        overlap / len(t)
    )

    return (
        2 * precision * recall
        / (precision + recall)
    )


def answer_similarity_score(pred, truth):
    return SequenceMatcher(
        None,
        normalized_answer(pred),
        normalized_answer(truth)
    ).ratio()


def context_similarity_score(
    answer,
    evidence
):
    if not str(answer).strip():
        return 0.0

    if not evidence:
        return 0.0

    model = get_embedder()

    chunks = []

    for item in evidence:
        if isinstance(item, dict):
            text = item.get(
                "document",
                item.get("content", "")
            )
        else:
            text = str(item)

        if str(text).strip():
            chunks.append(
                str(text)
            )

    if not chunks:
        return 0.0

    vectors = model.encode(
        [str(answer)] + chunks,
        normalize_embeddings=True
    )

    answer_vec = vectors[0]

    scores = np.dot(
        vectors[1:],
        answer_vec
    )

    return float(
        np.max(scores)
    )


def run_answer_evaluation(
    df,
    sleep_seconds=1.5,
    max_retries=4
):
    rows = []

    for i, row in df.iterrows():

        question = str(
            row[QUESTION_COL]
        ).strip()

        truth = str(
            row[ANSWER_COL]
        ).strip()

        result = None

        for attempt in range(
            max_retries
        ):
            try:
                result = run_rag_answer(
                    question
                )
                break

            except Exception as e:

                msg = str(e).lower()

                if (
                    "429" not in msg
                    and "rate limit" not in msg
                ) or attempt == max_retries - 1:

                    print(
                        f"Error for question "
                        f"{i + 1}: {e}"
                    )

                    break

                time.sleep(
                    sleep_seconds
                    * (attempt + 1)
                )

        if result is None:
            pred = ""
            context = ""
            evidence = []
        else:
            pred = result.get(
                "answer",
                ""
            )

            context = result.get(
                "context",
                ""
            )

            evidence = result.get(
                "evidence",
                []
            )

        rows.append({
            "Question": question,
            "Ground Truth": truth,
            "Generated Answer": pred,
            "Token F1": token_f1_score(
                pred,
                truth
            ),
            "Answer Similarity": answer_similarity_score(
                pred,
                truth
            ),
            "Context Similarity": context_similarity_score(
                pred,
                evidence
            ),
            "Question Type": detect_question_type(
                question
            )
        })

        print(
            f"Evaluated answer "
            f"{i + 1}/{len(df)}"
        )

        if (
            sleep_seconds > 0
            and i < len(df) - 1
        ):
            time.sleep(
                sleep_seconds
            )

    return pd.DataFrame(
        rows
    )


answer_results = run_answer_evaluation(
    ground_truth_df
)

display(
    answer_results.head()
)

print("\n" + "=" * 80)
print("ANSWER + CONTEXT EVALUATION SUMMARY")
print("=" * 80)

print(
    "Token F1          :",
    round(
        answer_results["Token F1"].mean(),
        4
    )
)

print(
    "Answer Similarity :",
    round(
        answer_results["Answer Similarity"].mean(),
        4
    )
)

print(
    "Context Similarity:",
    round(
        answer_results["Context Similarity"].mean(),
        4
    )
)


## 12. When a newer circular supersedes this one

Run this later, once you've also ingested the newer document.

In [ ]:
# mark_superseded(old_doc_id="BB-CRM-2016", new_doc_id="BB-CRM-2024")

## Final testing workflow

1. Run the notebook from top to bottom.
2. If the Chroma database was created with an older version of the chunk metadata,
   clear/recreate the vector database and re-ingest all Bangladesh Bank PDFs.
3. Upload the evaluation Excel with exactly these columns:
   `Question`, `Answer`, `PDF`, `Page`, `Section`.
4. Interactive retrieval uses:
   **query expansion + BM25 + BGE dense retrieval + RRF + lexical/metadata
   scoring + CrossEncoder reranking**.
5. Reasoning/scenario/comparison/calculation questions retrieve a wider evidence set
   and the CrossEncoder is given less authority than the hybrid retrieval signals.
6. Use `debug_retrieval(question)` before evaluating a difficult question. Check:
   PDF, section, page, BM25 score, dense score, RRF score, reranker score and final score.
7. The generation prompt first inspects all retrieved evidence and only returns
   "The provided documents do not contain this information." when the evidence
   genuinely does not support the answer.
8. Answer evaluation reports:
   **Token F1, Answer Similarity, and Context Similarity**.
   Exact Match is intentionally removed.

### Important

The ground-truth Excel is used only for evaluation. It is NOT used by the
interactive RAG system to generate answers.
